In [1]:
import sys
#akshare and xgboost
!{sys.executable} -m pip install -U akshare xgboost -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.0 MB/s  0:00:00 eta 0:00:01
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/79/98/679de17c2caa4fd3b0b4386ecf7377301702cb0afb22930a07c142fcb1d8/xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 10.6 MB/s  0:00:01m0:00:010:01
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/6c/dd/a834df6482147d48e225a49515aabc28974ad5a4ca3215c18a882565b028/html5lib-1.1-py2.py3-none-any.whl (112 kB)
  Using cached jsonpath-0.82.2-py3-none-any.whl
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/29/a9/8ce0ca222ef04d602924a1e099be93f5435ca6f3294182a30574d4159ca2/py_mini_racer-0.6.0-py2.py3-none-manylinux1_x86_64.whl (5.4 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/53/cb/1041355b14cd4b76ac082e8c676858f6eddb78f0ba37c59284adf36e5103/akracer-0.0.14-py3-none-any.whl (10.1 MB)
   ━━

In [56]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, linregress
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    log_loss,
)


In [57]:
# =========================
# 0. Parameter settings
# =========================
H = 5                    # future 5 bars
TH = 0.035                # threshold: 3.5%



In [58]:
# =========================
# 1. Load main continuous contract daily K-line data & handle time
# =========================
def load_main_csv(csv_path):

    df = pd.read_csv(csv_path)

    # 1) Build datetime index from `date`
    df["datetime"] = pd.to_datetime(df["date"])
    df = df.sort_values("datetime").set_index("datetime")

    # 2) Use adjusted OHLC columns (adj_*) and keep volume / hold as is
    data = pd.DataFrame(index=df.index)
    data["open"]   = df["adj_open"].astype(float)
    data["high"]   = df["adj_high"].astype(float)
    data["low"]    = df["adj_low"].astype(float)
    data["close"]  = df["adj_close"].astype(float)
    data["volume"] = df["volume"].astype(float)
    data["hold"]   = df["hold"].astype(float)
    return data

data = load_main_csv("TA.csv")
print(data.head())

                   open         high          low        close    volume  \
datetime                                                                   
2014-01-02  6674.572649  6674.572649  6592.998567  6611.126141  323978.0   
2014-01-03  6558.556177  6587.560295  6536.803089  6544.054118  173350.0   
2014-01-06  6534.990332  6536.803089  6467.918309  6476.982096  238492.0   
2014-01-07  6489.671397  6489.671397  6404.471801  6431.663161  221352.0   
2014-01-08  6424.412132  6458.854522  6413.535588  6438.914191  150578.0   

                hold  
datetime              
2014-01-02  403860.0  
2014-01-03  407484.0  
2014-01-06  422160.0  
2014-01-07  437950.0  
2014-01-08  427392.0  


In [59]:
# =========================
# 2. Load main continuous contract weekly K-line data & handle time
# =========================
def load_main_csv(csv_path):

    df = pd.read_csv(csv_path)

    # 1) Build datetime index from `date`
    df["datetime"] = pd.to_datetime(df["date"])
    df = df.sort_values("datetime").set_index("datetime")
    return df


data2 = load_main_csv("TA_weekly.csv")
print(data2.head())

                  date         open         high          low        close  \
datetime                                                                     
2014-01-03  2014-01-03  6674.572649  6674.572649  6536.803089  6544.054118   
2014-01-10  2014-01-10  6534.990332  6536.803089  6350.089079  6420.786617   
2014-01-17  2014-01-17  6426.224889  6440.726948  6275.766027  6377.371831   
2014-01-24  2014-01-24  6364.642546  6390.101116  6215.528063  6310.088467   
2014-01-31  2014-01-31  6297.359182  6311.906936  6197.343370  6237.349695   

               volume      hold  
datetime                         
2014-01-03   497328.0  407484.0  
2014-01-10  1140766.0  403938.0  
2014-01-17   995580.0  411552.0  
2014-01-24  1236006.0  436554.0  
2014-01-31   586102.0  406450.0  


In [60]:
print(data[['open', 'high', 'low', 'close', 'volume', 'hold']].isna().sum())

open      0
high      0
low       0
close     0
volume    0
hold      0
dtype: int64


In [61]:
# =========================
# 3. Price-action-only features
# =========================
def build_daily_features(
    df,
    mom_windows=(2, 4, 8, 16),
    vol_windows=(2, 4, 8, 16, 32),
    ma_windows=(2, 4, 8, 16, 32),
    vol_ma_windows=(3, 5, 10, 20),
    hold_ma_windows=(3, 5, 10, 20),
):

    data = df.copy()

    # =========================
    # 1. Single-bar return & candlestick shape
    # =========================
    data["ret_1"] = data["close"].pct_change()

    # Candlestick shape: body, range, upper/lower shadows
    data["body"] = data["close"] - data["open"]
    data["range"] = data["high"] - data["low"]
    data["upper_shadow"] = data["high"] - data[["open", "close"]].max(axis=1)
    data["lower_shadow"] = data[["open", "close"]].min(axis=1) - data["low"]

    # Ratios (normalized by range)
    data["body_pct"] = data["body"] / data["range"].replace(0, np.nan)
    data["upper_shadow_pct"] = data["upper_shadow"] / data["range"].replace(0, np.nan)
    data["lower_shadow_pct"] = data["lower_shadow"] / data["range"].replace(0, np.nan)

    data["is_bull"] = (data["close"] > data["open"]).astype(int)
    data["is_bear"] = (data["close"] < data["open"]).astype(int)

    # =========================
    # 2. Momentum, volatility & moving-average factors
    # =========================

    # -------- 1. Momentum --------
    for n in mom_windows:
        col = f"mom_{n}"
        data[col] = data["close"].pct_change(n)

        # slope: speed of momentum change 
        data[f"{col}_slope"] = data[col].pct_change()

        # multi-period slopes 
        data[f"{col}_slope_3"] = data[col].pct_change(3)
        data[f"{col}_slope_5"] = data[col].pct_change(5)
        data[f"{col}_slope_10"] = data[col].pct_change(10)
        data[f"{col}_slope_25"] = data[col].pct_change(25)

        # second derivative 
        data[f"{col}_acc"] = data[col].diff().diff()
        # angle slope
        data[f"mom_{n}_angle"] = np.arctan(
            data[f"mom_{n}"].diff() / (data[f"mom_{n}"].shift(1).abs() + 1e-9)
        )

    # -------- 2. Volatility --------
    for n in vol_windows:
        col = f"vol_{n}"
        data[col] = data["ret_1"].rolling(n).std()

        # slope: change in volatility 
        data[f"{col}_slope"] = data[col].pct_change()

        # multi-period slopes
        data[f"{col}_slope_3"] = data[col].pct_change(3)
        data[f"{col}_slope_5"] = data[col].pct_change(5)
        data[f"{col}_slope_10"] = data[col].pct_change(10)
        data[f"{col}_slope_25"] = data[col].pct_change(25)

        # second derivative: volatility acceleration 
        data[f"{col}_acc"] = data[col].diff().diff()


    # -------- 3. Moving averages (MA) --------
    
    #WMA
    data["wma_5"] = data["close"].rolling(5).apply(lambda x: np.dot(x, np.arange(1, 6)) / 15, raw=False)
    data["wma_10"] = data["close"].rolling(10).apply(lambda x: np.dot(x, np.arange(1, 11)) / np.sum(np.arange(1, 11)), raw=False)
    for n in ma_windows:
        ma_col = f"ma_{n}"
        data[ma_col] = data["close"].rolling(n).mean()
        data[f"close_ma_{n}"] = data["close"] / data[ma_col] - 1
        
        # slope: percentage change of MA 
        data[f"{ma_col}_slope"] = data[ma_col].pct_change()

        # multi-period slopes 
        data[f"{ma_col}_slope_3"] = data[ma_col].pct_change(3)
        data[f"{ma_col}_slope_5"] = data[ma_col].pct_change(5)
        data[f"{ma_col}_slope_10"] = data[ma_col].pct_change(10)
        data[f"{ma_col}_slope_25"] = data[ma_col].pct_change(25)

        # second derivative 
        data[f"{ma_col}_acc"] = data[ma_col].diff().diff()

        # angle slope 
        data[f"ma_{n}_angle"] = np.arctan(
            data[ma_col].diff() / (data[ma_col].shift(1).abs() + 1e-9)
        )

    # =========================
    # 3. Volume & open-interest factors
    # =========================

    # 1. Volume returns
    data["vol_ret_1"] = data["volume"].pct_change()
    data["vol_ret_3"] = data["volume"].pct_change(3)
    data["vol_ret_5"] = data["volume"].pct_change(5)
    data["vol_ret_10"] = data["volume"].pct_change(10)
    data["vol_ret_20"] = data["volume"].pct_change(20)

    # 2. Volume MA & bias
    for n in vol_ma_windows:
        ma_col = f"vol_ma_{n}"
        data[ma_col] = data["volume"].rolling(n).mean()
        data[f"vol_ma_bias_{n}"] = data["volume"] / data[ma_col] - 1
    
    # 3. Volume-Holding MA & Bias: 计算成交量与持仓量偏差   
    for n in vol_ma_windows:
        ma_col = f"vol_hold_ma_{n}"
        data[ma_col] = data["hold"].rolling(n).mean()  
        data[f"vol_hold_ma_bias_{n}"] = data["volume"] / data[ma_col] - 1 

    # 4. Volume volatility
    for n in vol_ma_windows:
        data[f"vol_vol_{n}"] = data["volume"].pct_change().rolling(n).std()

    # 5. Volume–price divergence
    data["vol_price_div_1"] = data["volume"].pct_change() - data["close"].pct_change()
    data["vol_price_div_3"] = data["volume"].pct_change(3) - data["close"].pct_change(3)
    data["vol_price_div_5"] = data["volume"].pct_change(5) - data["close"].pct_change(5)

    # =========================
    # 4. Hold (open interest) features
    # =========================

    # 1. Hold returns
    data["hold_ret_1"] = data["hold"].pct_change()
    data["hold_ret_3"] = data["hold"].pct_change(3)
    data["hold_ret_5"] = data["hold"].pct_change(5)
    data["hold_ret_10"] = data["hold"].pct_change(10)
    data["hold_ret_20"] = data["hold"].pct_change(20)

    # 2. Hold MA & bias
    for n in hold_ma_windows:
        ma_col = f"hold_ma_{n}"
        data[ma_col] = data["hold"].rolling(n).mean()
        data[f"hold_ma_bias_{n}"] = data["hold"] / data[ma_col] - 1

    # 3. Hold volatility
    for n in hold_ma_windows:
        data[f"hold_vol_{n}"] = data["hold"].pct_change().rolling(n).std()

    # 4. Price–OI divergence
    data["price_oi_div_1"] = data["close"].pct_change() - data["hold"].pct_change()
    data["price_oi_div_3"] = data["close"].pct_change(3) - data["hold"].pct_change(3)
    data["price_oi_div_5"] = data["close"].pct_change(5) - data["hold"].pct_change(5)

    # 5. Volume / hold ratio
    data["volume_hold_ratio"] = data["volume"] / (data["hold"] + 1e-6)

    # 6. Volume–price & Hold_price correlation
    data["vp_ratio_5"] = (
        data["volume"].pct_change().rolling(5).corr(data["close"].pct_change())
    )
    data["vp_ratio_10"] = (
        data["volume"].pct_change().rolling(10).corr(data["close"].pct_change())
    )
    data["hp_ratio_5"] = (
        data["hold"].pct_change().rolling(5).corr(data["close"].pct_change())
    )
    data["hp_ratio_10"] = (
        data["hold"].pct_change().rolling(10).corr(data["close"].pct_change())
    )

    # =========================
    # 5. Combined price–volume–OI factors
    # =========================
    data["price_vol_confirm"] = np.sign(data["ret_1"]) * data["vol_ret_1"]
    data["price_oi_trend"] = np.sign(data["ret_1"]) * data["hold_ret_1"]
    data["absorption_ratio"] = data["volume"] / (data["body"].abs() + 1e-3)
    data["effort_result_body_abs"] = data["volume"] * data["body"].abs()
    data["pv_corr_5"] = data["ret_1"].rolling(5).corr(data["vol_ret_1"])
    data["po_corr_5"] = data["ret_1"].rolling(5).corr(data["hold_ret_1"])
    data["vo_corr_5"] = data["vol_ret_1"].rolling(5).corr(data["hold_ret_1"])
    # RSI
    delta = data["close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    data["RSI_14"] = 100 - (100 / (1 + rs))
    # Stochastic Oscillator
    data["stochastic_14"] = (data["close"] - data["low"].rolling(14).min()) / (data["high"].rolling(14).max() - data["low"].rolling(14).min()) * 100

    return data


data = build_daily_features(data)


In [62]:
print(data.describe())

              open         high          low        close        volume  \
count  2428.000000  2428.000000  2428.000000  2428.000000  2.428000e+03   
mean   4896.292155  4947.037890  4845.903302  4895.580084  1.376483e+06   
std     796.397511   804.453922   787.043938   796.836415  7.822321e+05   
min    2926.091442  2944.552586  2878.092465  2891.015267  4.000000e+00   
25%    4397.845643  4436.090462  4353.546319  4394.611247  8.144220e+05   
50%    4878.309794  4919.753921  4828.238656  4872.812644  1.260930e+06   
75%    5458.194622  5514.835072  5379.681582  5451.222691  1.775475e+06   
max    6840.855993  6873.815149  6802.403644  6851.842378  8.394082e+06   

               hold        ret_1         body        range  upper_shadow  \
count  2.428000e+03  2427.000000  2428.000000  2428.000000   2428.000000   
mean   1.232520e+06     0.000078    -0.712071   101.134588     25.912094   
std    4.525907e+05     0.015613    70.581092    56.365376     23.127049   
min    4.200000e+01 

In [63]:
print("剩余 NaN 数量：")
print(data.isna().sum().sum())

剩余 NaN 数量：
2317


In [64]:

import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

def check_missing_values(data: pd.DataFrame, valid_by_finite: bool = False) -> pd.DataFrame:
    """
    遍历所有列（因子），统计 NaN 和 inf（+inf/-inf）情况。
    同时给出：
      - 总 NaN / 总 Inf
      - 有效区间内 NaN / Inf
      - 有效区间外 NaN / Inf

    valid_by_finite:
      False: 有效区间边界沿用原逻辑（first/last non-NaN）
      True : 有效区间边界用 first/last finite（既非 NaN 也非 inf）
    """
    missing_summary = []

    for col in data.columns:
        s = data[col]

        is_nan = s.isna()
        is_inf = np.isinf(s.to_numpy(dtype=float, copy=False))
        is_nonfinite = is_nan.to_numpy() | is_inf  # 仅用于 valid_by_finite 计算边界

        total_nan = int(is_nan.sum())
        total_inf = int(is_inf.sum())

        # decide valid-range boundaries
        if valid_by_finite:
            finite_mask = ~is_nonfinite
            if finite_mask.any():
                finite_arr = finite_mask.to_numpy()
                first_pos = int(np.argmax(finite_arr))
                last_pos = int(np.where(finite_arr)[0][-1])
                first_valid_index = s.index[first_pos]
                last_valid_index = s.index[last_pos]
            else:
                first_valid_index = None
                last_valid_index = None
        else:
            first_valid_index = s.first_valid_index()
            last_valid_index = s.last_valid_index()

        # stats within/outside valid range
        if (first_valid_index is not None) and (last_valid_index is not None):
            in_range = s.loc[first_valid_index:last_valid_index]

            in_nan = int(in_range.isna().sum())
            in_inf = int(np.isinf(in_range.to_numpy(dtype=float, copy=False)).sum())

            out_nan = total_nan - in_nan
            out_inf = total_inf - in_inf
        else:
            in_nan = in_inf = 0
            out_nan, out_inf = total_nan, total_inf

        missing_summary.append(
            {
                "Factor": col,

                "Total NaN": total_nan,
                "Total Inf": total_inf,

                "NaN in Valid Range": in_nan,
                "Inf in Valid Range": in_inf,

                "NaN outside Valid Range": out_nan,
                "Inf outside Valid Range": out_inf,

                "First Valid Index": first_valid_index,
                "Last Valid Index": last_valid_index,
            }
        )

    missing_summary_df = pd.DataFrame(missing_summary)

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    pd.set_option("display.max_colwidth", None)

    return missing_summary_df



# 检查缺失值并返回详细信息
missing_data_details = check_missing_values(data)
print(missing_data_details)

# 如果需要保存到文件：
# missing_data_details.to_csv('missing_data_details.csv', index=False)


                     Factor  Total NaN  Total Inf  NaN in Valid Range  \
0                      open          0          0                   0   
1                      high          0          0                   0   
2                       low          0          0                   0   
3                     close          0          0                   0   
4                    volume          0          0                   0   
5                      hold          0          0                   0   
6                     ret_1          1          0                   0   
7                      body          0          0                   0   
8                     range          0          0                   0   
9              upper_shadow          0          0                   0   
10             lower_shadow          0          0                   0   
11                 body_pct          4          0                   4   
12         upper_shadow_pct          4          0  

In [65]:
import pandas as pd

import numpy as np
import pandas as pd

def fill_missing_values(data: pd.DataFrame):
    data = data.replace([np.inf, -np.inf], np.nan)

    # 获取每个因子的有效区间
    def get_valid_range(col):
        first_valid_index = data[col].first_valid_index()
        last_valid_index = data[col].last_valid_index()
        if pd.notna(first_valid_index) and pd.notna(last_valid_index):
            return data.loc[first_valid_index:last_valid_index, col]
        else:
            return data[col]  # 如果没有有效范围，返回整个列

    # ========== 4. 检验后, 特殊处理部分 ==========

    # 1) 这些 slope 类因子：改为用前值填补 inf
    special_prev2_cols = [
        "vol_2_slope", "vol_2_slope_3", "vol_2_slope_5", "vol_2_slope_10", "vol_2_slope_25",
        "mom_16_slope", "mom_16_slope_3", "mom_16_slope_5", "mom_16_slope_10", "mom_16_slope_25",
        "mom_8_slope", "mom_8_slope_3", "mom_8_slope_5", "mom_8_slope_10", "mom_8_slope_25",
        "mom_4_slope", "mom_4_slope_3", "mom_4_slope_5", "mom_4_slope_10", "mom_4_slope_25",
        "mom_2_slope", "mom_2_slope_3", "mom_2_slope_5", "mom_2_slope_10", "mom_2_slope_25",
    ]

    for c in special_prev2_cols:
        if c not in data.columns:
            continue
        valid_range = get_valid_range(c)
        if valid_range.size > 0:
            print(f"Before ffill {c}, NaN count: {data[c].isna().sum()}")
            data.loc[valid_range.index, c] = data.loc[valid_range.index, c].ffill()
            print(f"After  ffill {c}, NaN count: {data[c].isna().sum()}")

    # 2) `mom_2_slope_3` 和 `mom_4_slope_3`：前值填补 nan
    for c in ["mom_2_slope_3", "mom_4_slope_3"]:
        if c not in data.columns:
            continue
        valid_range = get_valid_range(c)
        if valid_range.size > 0:
            print(f"Before filling {c}, NaN count: {data[c].isna().sum()}")
            data[c] = data[c].fillna(method="ffill")
            print(f"After  filling {c}, NaN count: {data[c].isna().sum()}")

    # 3) 比率因子：前值填补 nan
    for c in ["body_pct", "upper_shadow_pct", "lower_shadow_pct"]:
        if c not in data.columns:
            continue
        valid_range = get_valid_range(c)
        if valid_range.size > 0:
            print(f"Before filling {c}, NaN count: {data[c].isna().sum()}")
            data[c] = data[c].fillna(method="ffill")
            print(f"After  filling {c}, NaN count: {data[c].isna().sum()}")

    # 4) 删除所有因子中“最晚开始有有效值”之前的行（确保所有因子都有有效值）
    first_pos = []
    for c in data.columns:
        m = data[c].notna().to_numpy()
        if m.any():
            first_pos.append(int(m.argmax()))
    max_leading = max(first_pos) if first_pos else 0

    return data.iloc[max_leading:].copy()



# 假设你已经加载了数据，并且 `data` 是一个有效的 DataFrame
data = fill_missing_values(data)


Before ffill vol_2_slope, NaN count: 10
After  ffill vol_2_slope, NaN count: 3
Before ffill vol_2_slope_3, NaN count: 12
After  ffill vol_2_slope_3, NaN count: 5
Before ffill vol_2_slope_5, NaN count: 14
After  ffill vol_2_slope_5, NaN count: 7
Before ffill vol_2_slope_10, NaN count: 19
After  ffill vol_2_slope_10, NaN count: 12
Before ffill vol_2_slope_25, NaN count: 34
After  ffill vol_2_slope_25, NaN count: 27
Before ffill mom_16_slope, NaN count: 23
After  ffill mom_16_slope, NaN count: 17
Before ffill mom_16_slope_3, NaN count: 25
After  ffill mom_16_slope_3, NaN count: 19
Before ffill mom_16_slope_5, NaN count: 27
After  ffill mom_16_slope_5, NaN count: 21
Before ffill mom_16_slope_10, NaN count: 32
After  ffill mom_16_slope_10, NaN count: 26
Before ffill mom_16_slope_25, NaN count: 47
After  ffill mom_16_slope_25, NaN count: 41
Before ffill mom_8_slope, NaN count: 16
After  ffill mom_8_slope, NaN count: 9
Before ffill mom_8_slope_3, NaN count: 18
After  ffill mom_8_slope_3, NaN 

In [66]:
missing_data_details = check_missing_values(data)
print(missing_data_details)

                     Factor  Total NaN  Total Inf  NaN in Valid Range  \
0                      open          0          0                   0   
1                      high          0          0                   0   
2                       low          0          0                   0   
3                     close          0          0                   0   
4                    volume          0          0                   0   
5                      hold          0          0                   0   
6                     ret_1          0          0                   0   
7                      body          0          0                   0   
8                     range          0          0                   0   
9              upper_shadow          0          0                   0   
10             lower_shadow          0          0                   0   
11                 body_pct          0          0                   0   
12         upper_shadow_pct          0          0  

In [67]:
# =========================
# 5. Construct labels
# =========================

# 1. Close-to-close return over horizon H
data["future_ret_H"] = data["close"].shift(-H) / data["close"] - 1

# Binary labels for large upward / downward moves
data["y_up"] = (data["future_ret_H"] >= TH).astype(int)
data["y_down"] = (data["future_ret_H"] <= -TH).astype(int)

# 4. Drop samples whose future returns are not observable
mask = data["future_ret_H"].notna()
data_train = data[mask].copy()


In [68]:
# ============================================
# 6. Train/Test split
# ============================================

data_raw_for_backtest = data_train.copy()

feature_cols = [
    c for c in data_train.columns
    if not c.startswith("future_")
    and c not in ["y_up", "y_down"]
]

train_ratio = 0.6
train_size = int(len(data_train) * train_ratio)

train_all = data_train.iloc[:train_size].copy()
test      = data_train.iloc[train_size:].copy()

print("Train:", len(train_all), "| Test:", len(test))


Train: 1419 | Test: 947


In [69]:
print("剩余 NaN 数量：")
print(train_all.isna().sum().sum())

剩余 NaN 数量：
0


In [70]:
# ============================================
# 7. Preprocessing
# ============================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.stats import kurtosis


import numpy as np
import pandas as pd
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler, RobustScaler


import numpy as np
import pandas as pd
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler, RobustScaler


class FeaturePreprocessor:
    def __init__(
        self,
        heavy_kurtosis=6,
        clip_pct=(1, 99),
        log_candidates=("volume", "hold"),
        sigma_clip=6,
        robust=False,
        check_finite=True,
    ):
        self.heavy_kurtosis = heavy_kurtosis
        self.clip_low, self.clip_high = clip_pct
        self.log_candidates = tuple(log_candidates)
        self.sigma_clip = sigma_clip
        self.use_robust = robust
        self.check_finite = check_finite

        self.feature_cols = None
        self.heavy_cols = []

        # heavy cols percentile bounds: {col: (lo, hi)}
        self.clip_bounds = {}

        # sigma-clip bounds for cols: {col: (mean, std)}
        self.sigma_bounds = {}

        self.scaler = None
        self.is_fitted = False

    # ============================
    # utils
    # ============================
    def _check_no_nan_inf(self, X: pd.DataFrame, where: str):
        if not self.check_finite:
            return

        arr = X.to_numpy(dtype=float, copy=False)
        if np.isfinite(arr).all():
            return

        bad = ~np.isfinite(arr)
        i, j = np.argwhere(bad)[0]
        col = X.columns[j]
        idx = X.index[i]
        v = X.iloc[i, j]

        raise ValueError(
            f"[{where}] found non-finite value at index={idx}, col='{col}', value={v}. "
            "This preprocessor does NOT handle NaN/inf; please clean data before calling."
        )

    # ============================
    # 1) detect heavy-tail columns
    # ============================
    def _detect_heavy(self, X: pd.DataFrame):
        heavy = []
        for c in self.feature_cols:
            if ("slope" in c) or ("ret" in c):
                continue

            x = X[c].to_numpy(dtype=float, copy=False)
            try:
                k = kurtosis(x, fisher=False, bias=False)
                if np.isfinite(k) and (k > self.heavy_kurtosis):
                    heavy.append(c)
            except Exception:
                # 保守：检测失败就不自动归为 heavy
                pass

        return heavy

    # ============================
    # 2) signed-log with percentile clip (NO NaN handling)
    # ============================
    def _signed_log_clip(self, x: np.ndarray, col: str, fit: bool):
        x = np.asarray(x, dtype=float)

        if fit:
            lo, hi = np.percentile(x, [self.clip_low, self.clip_high])

            # 常数列/异常列兜底：避免 lo>=hi
            if (not np.isfinite(lo)) or (not np.isfinite(hi)) or (lo >= hi):
                m = float(np.mean(x))
                eps = 1e-12
                lo, hi = m - eps, m + eps

            self.clip_bounds[col] = (float(lo), float(hi))
        else:
            lo, hi = self.clip_bounds[col]

        x = np.clip(x, lo, hi)
        return np.sign(x) * np.log1p(np.abs(x))

    # ============================
    # 3) sigma-clip for ALL columns (NO NaN handling)
    # ============================
    def _sigma_clip(self, x: np.ndarray, col: str, fit: bool):
        x = np.asarray(x, dtype=float)

        if fit:
            m = float(np.mean(x))
            sd = float(np.std(x)) + 1e-12
            self.sigma_bounds[col] = (m, sd)
        else:
            m, sd = self.sigma_bounds[col]

        lo = m - self.sigma_clip * sd
        hi = m + self.sigma_clip * sd
        return np.clip(x, lo, hi)

    # ============================
    # 4) fit(train)
    # ============================
    def fit(self, df: pd.DataFrame, feature_cols):
        """
        假设 df[feature_cols] 已经在外部处理完 NaN/inf，这里只做：
        1) heavy: percentile clip + signed-log（训练期学边界）
        2) all features: 均值±sigma_clip*std 的 σ-clip（训练期学 m, sd）
        3) scaler fit（训练期）
        """
        assert df.index.is_monotonic_increasing, "df must be sorted in ascending time order"
        assert df.index.is_unique, "df index must be unique"
        assert isinstance(df.index, pd.DatetimeIndex), "df.index must be a DatetimeIndex"

        self.feature_cols = list(feature_cols)
        X = df[self.feature_cols].copy()

        self._check_no_nan_inf(X, where="fit")

        # ===== heavy cols: auto + manual =====
        auto_heavy = self._detect_heavy(X)
        manual_heavy = [c for c in self.log_candidates if c in self.feature_cols]
        self.heavy_cols = sorted(set(auto_heavy + manual_heavy))

        # ===== apply transforms on TRAIN to learn bounds =====
        Xp = X.copy()

        for col in self.heavy_cols:
            Xp[col] = self._signed_log_clip(Xp[col].to_numpy(copy=False), col, fit=True)

        for col in self.feature_cols:
            Xp[col] = self._sigma_clip(Xp[col].to_numpy(copy=False), col, fit=True)

        self.scaler = RobustScaler() if self.use_robust else StandardScaler()
        self.scaler.fit(Xp[self.feature_cols])

        self.is_fitted = True
        return self

    # ============================
    # 5) transform(test/val)
    # ============================
    def transform(self, df: pd.DataFrame):
        if not self.is_fitted:
            raise RuntimeError("Call fit(train) before transform(test).")
        
        assert df.index.is_monotonic_increasing, "df must be sorted in ascending time order"
        assert df.index.is_unique, "df index must be unique"
        assert isinstance(df.index, pd.DatetimeIndex), "df.index must be a DatetimeIndex"

        X = df[self.feature_cols].copy()
        self._check_no_nan_inf(X, where="transform")

        Xp = X.copy()

        for col in self.heavy_cols:
            Xp[col] = self._signed_log_clip(Xp[col].to_numpy(copy=False), col, fit=False)

        for col in self.feature_cols:
            Xp[col] = self._sigma_clip(Xp[col].to_numpy(copy=False), col, fit=False)

        Z = self.scaler.transform(Xp[self.feature_cols])

        out = df.copy()
        out[self.feature_cols] = Z
        return out

    def report(self, df: pd.DataFrame):
        X = df[self.feature_cols]
        stats = X.describe().T
        print(stats[["mean", "std", "min", "max"]])
        print("\nMax abs:", X.abs().to_numpy().max())


In [71]:
# ============================================
# 8. Feature preprocessing: fit(train) + transform(train/test)
# ============================================
# 1) Keep only feature columns
X_train = train_all[feature_cols]
X_test  = test[feature_cols]

# 2) Preprocessor
prep = FeaturePreprocessor()

# 3) Fit ONLY on X_train 
prep.fit(X_train, feature_cols)

# 4) Transform train/test (using statistics from train only)
X_train_scaled = prep.transform(X_train).sort_index()
X_test_scaled  = prep.transform(X_test).sort_index()

# 5) Sanity check: ensure no inf/nan 
assert np.all(np.isfinite(X_train_scaled.values)), "Train contains inf/nan!"
assert np.all(np.isfinite(X_test_scaled.values)),  "Test contains inf/nan!"

# 6) Attach labels back 
train_all_scaled = X_train_scaled.copy()
train_all_scaled["y_up"]   = train_all["y_up"]
train_all_scaled["y_down"] = train_all["y_down"]

test_scaled = X_test_scaled.copy()
test_scaled["y_up"]   = test["y_up"]
test_scaled["y_down"] = test["y_down"]


In [72]:
df_scaled = train_all_scaled[feature_cols]

# 1) Which features have the largest absolute values?
max_abs_per_col = df_scaled.abs().max().sort_values(ascending=False)
print("=== Top 10 features by max |z| (descending) ===")
print(max_abs_per_col.head(10))

# 2) On which dates do extreme values occur?
max_abs_per_row = df_scaled.abs().max(axis=1).sort_values(ascending=False)
print("\n=== Top 10 dates with the largest row-wise max |z| ===")
print(max_abs_per_row.head(10))

# Inspect the original factor values on one extreme date
extreme_idx = max_abs_per_row.index[0]
print("\nExtreme date:", extreme_idx)
print("Feature z-values on this date:")
print(df_scaled.loc[extreme_idx].sort_values(key=np.abs, ascending=False).head(10))

print("\nOriginal (unscaled) data on this date:")
print(train_all.loc[extreme_idx].head(20))

=== Top 10 features by max |z| (descending) ===
vol_2_slope      23.892385
hold_ret_20      23.606250
vol_2_slope_5    22.676380
hold_ret_1       21.725428
hold_ret_3       21.725423
hold_ret_5       21.725403
hold_ret_10      21.725297
vol_ret_1        21.627232
vol_ret_10       21.353085
vol_2_slope_3    21.343796
dtype: float64

=== Top 10 dates with the largest row-wise max |z| ===
datetime
2019-07-01    23.892385
2016-12-09    23.892385
2015-11-05    23.606250
2014-05-30    23.606250
2016-12-15    22.676380
2017-01-25    22.676380
2015-10-09    21.725428
2017-10-10    21.725428
2014-05-05    21.725428
2014-05-07    21.725423
dtype: float64

Extreme date: 2019-07-01 00:00:00
Feature z-values on this date:
vol_2_slope    23.892385
ret_1           4.565264
close_ma_16     4.258135
close_ma_8      4.100176
mom_4           4.094375
vol_2_acc       4.006038
ma_4_slope      3.970993
ma_4_angle      3.970335
mom_16          3.884843
close_ma_4      3.795647
Name: 2019-07-01 00:00:00, dtyp

In [73]:
# ============================================
# 9. Train on XGB
# ============================================

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, classification_report


def train_one_direction_xgb(
    df,
    feature_cols,
    target_col,
    direction_name="UP",
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=50,   # -> 映射到 XGB 的 min_child_weight（越大越保守）
    max_features="sqrt",   # -> 映射到 XGB 的 colsample_bytree（每棵树采样的特征比例）

    # ===== OOS split params =====
    valid_size=250,
    n_windows=3,           # 从尾部切 n_windows 个连续 valid block（每块 valid_size）

    # ===== Feature selection params =====
    imp_min=0.0,           # OOS permutation importance 的绝对阈值（可设 0）
    imp_quantile=0.6,      # OOS importance 分位阈值（例如 0.6 表示保留前 40%）
    top_k_max=25,          # 最终最多保留多少个特征
    auc_single_min=0.62,   # 单因子 OOS AUC 均值阈值
    n_perm=5,              # permutation 次数（越大越稳但越慢）
    max_single_check=25,   # 最多对多少个候选特征做“单因子AUC复核”
):
    """
    多窗口时间序列 Walk-Forward（expanding train / forward valid）：
    1) 每个窗口：用 train 拟合 XGB
    2) 在 valid 上做 OOS permutation importance（打乱单个特征列，观察性能下降）
    3) 聚合多个窗口的 OOS importance（mean/median/pos_frac）
    4) 重要性过阈值后，对 top-N 候选特征做“单因子 XGB”的 OOS AUC 复核
    5) 输出最终选中的特征、重要性表、以及最后一个窗口训练出来的 FINAL model

    注意：
    - 该函数内部切分遵循时间顺序：train 永远在 valid 之前（避免训练阶段泄漏）。
    - 但最后输出的 prob_all 是“用最后窗口训练的模型”对全样本打分，适合诊断/画图；
      若要严格回测，请用 walk-forward 的 OOS 预测序列（需要另外实现 oof_prob 填充）。
    """

    print(f"\n========== Start training {direction_name} (XGB), target = {target_col} ==========\n")

    # ==========================================================
    # 0) 时间排序 + 基本合法性检查
    # ==========================================================
    df = df.sort_index()          # 强制按时间升序
    n = len(df)
    if n <= valid_size:
        raise ValueError(f"Need n > valid_size. Got n={n}, valid_size={valid_size}.")

    # 最大可用窗口数：保证最早窗口至少有 1 条 train
    max_w = (n - 1) // valid_size
    if n_windows > max_w:
        print(
            f"[Warn] n_windows={n_windows} too large for n={n}, valid_size={valid_size}. "
            f"Capping to {max_w}."
        )
        n_windows = max_w

    # ==========================================================
    # 1) 从尾部构造 n_windows 个 (train, valid) 窗口
    #    - valid 是连续的一段长度 valid_size
    #    - train 是 valid 之前的全部历史（expanding）
    # ==========================================================
    windows = []
    for w in range(n_windows, 0, -1):
        train_end = n - w * valid_size         # train: [0, train_end)
        valid_start = train_end
        valid_end = train_end + valid_size     # valid: [valid_start, valid_end)
        if train_end <= 0:
            continue
        windows.append((0, train_end, valid_start, valid_end))

    # 这里 windows 已经是从“更早窗口 -> 更晚窗口”（因为 w 从大到小 append）
    # windows = windows  # 这行本身是冗余的，占位说明顺序无需 reverse

    if len(windows) == 0:
        raise RuntimeError("Failed to construct windows. Check n/valid_size/n_windows.")

    print(f"[Info] Using {len(windows)} window(s):")
    for k, (tr0, tr1, va0, va1) in enumerate(windows, 1):
        print(f"  W{k}: train[{tr0}:{tr1}] ({tr1 - tr0}), valid[{va0}:{va1}] ({va1 - va0})")

    # ==========================================================
    # 2) max_features -> colsample_bytree（特征子采样比例）
    # ==========================================================
    d = len(feature_cols)
    if isinstance(max_features, str):
        if max_features == "sqrt":
            # 每棵树采样 sqrt(d) 个特征 -> 比例 sqrt(d)/d
            colsample_bytree = float(np.sqrt(d) / d) if d > 0 else 1.0
        elif max_features == "log2":
            colsample_bytree = float(np.log2(d) / d) if d > 0 else 1.0
        else:
            colsample_bytree = 1.0
    else:
        # 若用户直接给比例（如 0.7），直接用
        colsample_bytree = float(max_features)

    # 特征名 -> 列索引（用于单因子训练时取 [:, [j]]）
    col_to_j = {c: i for i, c in enumerate(feature_cols)}

    # ==========================================================
    # 3) 统一 metric_mode：所有窗口都可算 AUC 才用 AUC，否则用 -logloss
    # ==========================================================
    def has_both_classes(y_arr) -> bool:
        """检查标签是否同时包含 0 和 1（AUC 需要双类）。"""
        return len(np.unique(y_arr)) > 1

    use_auc = True
    for (_, _, va0, va1) in windows:
        y_va = df.iloc[va0:va1][target_col].to_numpy().astype(int)
        if not has_both_classes(y_va):
            use_auc = False
            break

    metric_mode = "auc" if use_auc else "neg_logloss"
    print(f"\n[Info] metric_mode for permutation importance = {metric_mode}")

    def metric_score(y_true, proba_2col):
        """
        统一“越大越好”的分数：
        - auc：直接用 AUC
        - neg_logloss：用 -logloss（取负号使得越大越好）
        """
        if metric_mode == "auc":
            return roc_auc_score(y_true, proba_2col[:, 1])
        else:
            return -log_loss(y_true, proba_2col, labels=[0, 1])

    def build_model(scale_pos_weight: float, random_state: int):
        """构建一个 XGBClassifier（统一超参），并传入类别不平衡权重与随机种子。"""
        return XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=colsample_bytree,
            reg_lambda=1.0,
            min_child_weight=float(min_samples_leaf),  # RF 的 min_samples_leaf -> XGB 的 min_child_weight
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=random_state,
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
        )

    # ==========================================================
    # 4) 多窗口：训练模型 + 在 valid 上做 OOS permutation importance
    # ==========================================================
    imp_by_window = []     # 每个窗口一个 Series(perm_importance)
    window_metrics = []    # 每个窗口的 valid 指标（ACC/AUC/baseline等）

    for w, (tr0, tr1, va0, va1) in enumerate(windows, 1):
        # --- 切分 train/valid（按时间顺序）
        train_df = df.iloc[tr0:tr1]
        valid_df = df.iloc[va0:va1]

        # --- 取特征与标签
        X_tr = train_df[feature_cols].to_numpy(dtype=np.float32)
        y_tr = train_df[target_col].to_numpy().astype(int)
        X_va = valid_df[feature_cols].to_numpy(dtype=np.float32)
        y_va = valid_df[target_col].to_numpy().astype(int)

        # --- 类别不平衡处理：scale_pos_weight = neg/pos
        pos_ratio_tr = y_tr.mean() if len(y_tr) else np.nan
        n_pos = (y_tr == 1).sum()
        n_neg = (y_tr == 0).sum()
        scale_pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0

        # --- 训练该窗口模型（仅用 train）
        model_w = build_model(scale_pos_weight=scale_pos_weight, random_state=2025 + 17 * w)
        model_w.fit(X_tr, y_tr)

        # --- valid 上的 baseline 预测与分数
        proba_va = model_w.predict_proba(X_va)          # shape = (n_valid, 2)
        baseline = metric_score(y_va, proba_va)         # AUC 或 -logloss

        # --- 诊断指标：ACC/AUC（AUC 可能因为单类而 NaN）
        prob_va = proba_va[:, 1]
        pred_va = (prob_va > 0.5).astype(int)
        acc_va = accuracy_score(y_va, pred_va)
        auc_va = roc_auc_score(y_va, prob_va) if has_both_classes(y_va) else np.nan

        # --- 记录窗口表现
        window_metrics.append({
            "window": w,
            "train_len": len(train_df),
            "valid_len": len(valid_df),
            "pos_ratio_train": pos_ratio_tr,
            "acc_valid": acc_va,
            "auc_valid": auc_va,
            "baseline_metric": baseline,
            "metric_mode": metric_mode,
        })

        # --- permutation importance：对每个特征列打乱，观察性能下降
        rng = np.random.RandomState(2025 + 1000 * w)
        perm_importance = {}

        for j, col in enumerate(feature_cols):
            deltas = []
            for _ in range(n_perm):
                # 复制 valid 特征矩阵
                Xp = X_va.copy()

                # 打乱第 j 列（仅在副本上做，不改变原数据）
                shuffled = Xp[:, j].copy()
                rng.shuffle(shuffled)
                Xp[:, j] = shuffled

                # 重新预测 + 重新算分
                proba_p = model_w.predict_proba(Xp)
                metric_p = metric_score(y_va, proba_p)

                # baseline - metric_p = 性能下降量（越大说明该特征越重要）
                deltas.append(baseline - metric_p)

            perm_importance[col] = float(np.mean(deltas))

        # 当前窗口的重要性（从大到小排序）
        imp_ser = pd.Series(perm_importance).sort_values(ascending=False)
        imp_by_window.append(imp_ser)

        # --- 打印窗口结果（便于调参/诊断）
        print(f"\n--- W{w} OOS valid metrics ---")
        print(
            f"train_len={len(train_df)}, valid_len={len(valid_df)}, pos_ratio_train={pos_ratio_tr:.4f}, "
            f"valid_ACC={acc_va:.4f}, valid_AUC={auc_va:.4f}"
        )
        print(f"W{w} OOS permutation importance (Top 10):")
        print(imp_ser.head(10))

    # ==========================================================
    # 4b) 聚合多个窗口的 importance（mean/median/pos_frac）
    # ==========================================================
    imp_mat = pd.concat(imp_by_window, axis=1)   # 行=特征，列=窗口
    imp_mean = imp_mat.mean(axis=1)
    imp_median = imp_mat.median(axis=1)
    imp_pos_frac = (imp_mat > 0).mean(axis=1)    # importance>0 的窗口比例（稳定性）

    importance_series = imp_mean.sort_values(ascending=False)

    print(f"\n[{direction_name}] Aggregated OOS permutation importance (mean) Top 20:")
    top20 = importance_series.head(20).index
    print(pd.DataFrame({
        "imp_mean": imp_mean.loc[top20],
        "imp_median": imp_median.loc[top20],
        "pos_frac": imp_pos_frac.loc[top20],
    }))

    # ==========================================================
    # 5) 用“最后一个窗口”的 train 训练 FINAL 模型，并在其 valid 上做最终 OOS 输出
    # ==========================================================
    tr0, tr1, va0, va1 = windows[-1]
    train_df = df.iloc[tr0:tr1]
    valid_df = df.iloc[va0:va1]

    X_tr = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_tr = train_df[target_col].to_numpy().astype(int)
    X_va = valid_df[feature_cols].to_numpy(dtype=np.float32)
    y_va = valid_df[target_col].to_numpy().astype(int)

    n_pos = (y_tr == 1).sum()
    n_neg = (y_tr == 0).sum()
    scale_pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0

    model = build_model(scale_pos_weight=scale_pos_weight, random_state=2025)
    model.fit(X_tr, y_tr)

    proba_va = model.predict_proba(X_va)
    prob_va = proba_va[:, 1]
    pred_va = (prob_va > 0.5).astype(int)

    acc = accuracy_score(y_va, pred_va)
    auc = roc_auc_score(y_va, prob_va) if has_both_classes(y_va) else np.nan

    print(f"\n{direction_name} - FINAL OOS (train={len(train_df)}/valid={len(valid_df)}) ACC: {acc:.4f}")
    print(f"{direction_name} - FINAL OOS AUC:  {auc:.4f}")
    print(f"\n{direction_name} - FINAL OOS classification_report:")
    print(classification_report(y_va, pred_va))

    # XGB 内置重要性（参考用：不如 OOS permutation 稳健）
    gini_series = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

    # ==========================================================
    # 6) 基于聚合 OOS importance 做第一轮筛选
    # ==========================================================
    thr_abs = -np.inf if (imp_min is None) else imp_min
    thr_quantile = -np.inf if (imp_quantile is None) else imp_mean.quantile(imp_quantile)
    thr = max(thr_abs, thr_quantile)

    print(
        f"\n{direction_name} - OOS perm-imp threshold: max(imp_min={thr_abs:.6f}, "
        f"quantile={thr_quantile:.6f}) = {thr:.6f}"
    )

    # 通过阈值的候选特征（按 imp_mean 降序）
    imp_candidates = imp_mean[imp_mean >= thr].sort_values(ascending=False).index.tolist()
    print(f"{direction_name} - features passing OOS importance threshold: {len(imp_candidates)}")

    # 单因子复核只做 top max_single_check（节约时间）
    single_check_features = (
        imp_candidates[:max_single_check] if (max_single_check is not None) else imp_candidates
    )
    print(
        f"{direction_name} - Running single-factor OOS AUC check for top {len(single_check_features)} features"
    )

    # ==========================================================
    # 7) 单因子 OOS AUC 复核（跨窗口取均值/最小值）
    # ==========================================================
    single_auc_by_window = {col: [] for col in single_check_features}

    if (auc_single_min is not None) and (len(single_check_features) > 0):
        print(f"\n{direction_name} - Single-factor OOS AUC check (threshold = {auc_single_min:.3f})")

        for col in single_check_features:
            j = col_to_j[col]  # 该特征在 feature_cols 中的列索引

            for w, (tr0, tr1, va0, va1) in enumerate(windows, 1):
                train_df_w = df.iloc[tr0:tr1]
                valid_df_w = df.iloc[va0:va1]

                # 只取单列特征，保持二维形状 (n, 1)
                X_tr_w = train_df_w[feature_cols].to_numpy(dtype=np.float32)[:, [j]]
                y_tr_w = train_df_w[target_col].to_numpy().astype(int)
                X_va_w = valid_df_w[feature_cols].to_numpy(dtype=np.float32)[:, [j]]
                y_va_w = valid_df_w[target_col].to_numpy().astype(int)

                # 类别不平衡权重
                n_pos_w = (y_tr_w == 1).sum()
                n_neg_w = (y_tr_w == 0).sum()
                spw_w = (n_neg_w / max(n_pos_w, 1)) if n_pos_w > 0 else 1.0

                # 单因子模型（简单些：树数上限 200，colsample=1.0）
                single_xgb = XGBClassifier(
                    n_estimators=min(200, n_estimators),
                    max_depth=max_depth,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=1.0,
                    reg_lambda=1.0,
                    min_child_weight=float(min_samples_leaf),
                    objective="binary:logistic",
                    eval_metric="logloss",
                    n_jobs=-1,
                    random_state=2025 + 31 * w + 7 * j,
                    scale_pos_weight=spw_w,
                    tree_method="hist",
                )

                try:
                    single_xgb.fit(X_tr_w, y_tr_w)
                    proba = single_xgb.predict_proba(X_va_w)
                    auc_w = roc_auc_score(y_va_w, proba[:, 1]) if has_both_classes(y_va_w) else np.nan
                except Exception:
                    # 某些窗口可能因为单类/数值问题训练失败，用 NaN 占位
                    auc_w = np.nan

                single_auc_by_window[col].append(auc_w)

        # 跨窗口聚合：均值与最小值（最小值可用于衡量稳定性）
        single_auc_mean = pd.Series({
            col: float(np.nanmean(aucs)) if len(aucs) else np.nan
            for col, aucs in single_auc_by_window.items()
        }).sort_values(ascending=False)

        single_auc_min = pd.Series({
            col: float(np.nanmin(aucs)) if np.any(~np.isnan(aucs)) else np.nan
            for col, aucs in single_auc_by_window.items()
        })

        print(f"\n{direction_name} - Single-factor OOS AUC mean (Top 20):")
        top20 = single_auc_mean.head(20).index
        print(pd.DataFrame({
            "auc_mean": single_auc_mean.loc[top20],
            "auc_min": single_auc_min.loc[top20],
        }))
    else:
        # 未启用单因子筛选：返回全 NaN
        single_auc_mean = pd.Series({col: np.nan for col in feature_cols})
        single_auc_min = pd.Series({col: np.nan for col in feature_cols})

    # ==========================================================
    # 8) 最终特征选择：在 base_candidates 中按单因子 AUC 阈值筛选，再截断 top_k
    # ==========================================================
    base_candidates = single_check_features

    if auc_single_min is not None:
        selected_features = [
            col for col in base_candidates
            if (col in single_auc_mean.index)
            and (not np.isnan(single_auc_mean[col]))
            and (single_auc_mean[col] >= auc_single_min)
        ]
    else:
        selected_features = list(base_candidates)

    if (top_k_max is not None) and (len(selected_features) > top_k_max):
        selected_features = selected_features[:top_k_max]

    print(f"\n{direction_name} - Final selected features: {len(selected_features)}")
    print(f"{direction_name} - Selected list (top 30 shown):")
    print(selected_features[:30])

    # ==========================================================
    # 9) 用 FINAL model 对全 df 打分（注意：严格回测不建议直接用它做全样本信号）
    # ==========================================================
    X_all = df[feature_cols].to_numpy(dtype=np.float32)
    y_all = df[target_col].to_numpy().astype(int)
    proba_all = model.predict_proba(X_all)
    prob_all = proba_all[:, 1]
    pred_all = (prob_all > 0.5).astype(int)

    # ==========================================================
    # 结果打包：便于后续画图/调参/接 score/backtest
    # ==========================================================
    result = {
        "model": model,
        "top_features": selected_features,         # 兼容你之前的字段命名
        "selected_features": selected_features,

        "importance_series": importance_series,    # 按 imp_mean 排序的 Series
        "importance_detail": pd.DataFrame({
            "imp_mean": imp_mean,
            "imp_median": imp_median,
            "pos_frac": imp_pos_frac,
        }).sort_values("imp_mean", ascending=False),

        "gini_importance": gini_series,            # 内置重要性（参考）
        "single_auc": single_auc_mean,             # 单因子 AUC 均值

        "acc": acc,                                # 最后窗口 valid 的 ACC
        "auc": auc,                                # 最后窗口 valid 的 AUC

        "prob": prob_all,                          # 全样本概率（更多用于诊断）
        "pred": pred_all,
        "true": y_all,
        "index": df.index,

        "windows": windows,                        # 窗口切分边界
        "window_metrics": window_metrics,          # 每窗口 valid 指标
        "single_auc_min": single_auc_min,          # 单因子 AUC 最差窗口
    }

    return result


# TODO（后续扩展建议）：
# - 你加入 RF / LightGBM 时，可以复用：
#   1) windows 构造逻辑
#   2) OOS permutation importance 框架（baseline - permuted_score）
#   3) 单因子复核框架（把单因子模型换成 RF/LGB/XGB）
# - 若用于严格回测：建议新增 oof_prob（walk-forward 仅填 valid 段预测），避免 look-ahead。


In [74]:
# ============================================
# 10. Train XGB models for large up / down moves
# ============================================

up_result_lstm = train_one_direction_xgb(
    df=train_all_scaled,
    feature_cols=feature_cols,
    target_col="y_up",
    direction_name="UP (large up move ≥ threshold)",
    imp_min=0.0,
    imp_quantile=0.7,
    top_k_max=20,
    auc_single_min=0.5,
    n_perm=5,
    max_single_check=25,
)

down_result_lstm = train_one_direction_xgb(
    df=train_all_scaled,
    feature_cols=feature_cols,
    target_col="y_down",
    direction_name="DOWN (large down move ≤ -threshold)",
    imp_min=0.0,
    imp_quantile=0.7,
    top_k_max=20,
    auc_single_min=0.5,
    n_perm=5,
    max_single_check=25,
)

print("\nXGB model (UP) final selected factors:",   up_result_lstm["selected_features"])
print("XGB model (DOWN) final selected factors:",  down_result_lstm["selected_features"])

print("\nXGB model (UP) Top features (compatible name `top_features`):",
      up_result_lstm["top_features"])
print("XGB model (DOWN) Top features (compatible name `top_features`):",
      down_result_lstm["top_features"])



========== Start training UP (large up move ≥ threshold) (XGB), target = y_up ==========

[Info] Using 3 window(s):
  W1: train[0:669] (669), valid[669:919] (250)
  W2: train[0:919] (919), valid[919:1169] (250)
  W3: train[0:1169] (1169), valid[1169:1419] (250)

[Info] metric_mode for permutation importance = auc

--- W1 OOS valid metrics ---
train_len=669, valid_len=250, pos_ratio_train=0.1136, valid_ACC=0.8200, valid_AUC=0.4807
W1 OOS permutation importance (Top 10):
vol_32_slope_10    0.056955
mom_8_slope        0.020651
mom_8_slope_5      0.018127
vol_4_slope_10     0.016864
vol_4_slope_5      0.015299
vol_4              0.014239
vol_16             0.014188
ma_8_slope_10      0.013128
ma_16_slope_25     0.012977
vol_ret_10         0.012522
dtype: float64

--- W2 OOS valid metrics ---
train_len=919, valid_len=250, pos_ratio_train=0.1012, valid_ACC=0.8120, valid_AUC=0.5937
W2 OOS permutation importance (Top 10):
vol_32_slope_25    0.046952
vol_32             0.024095
vol_16_slope_25

In [75]:
# ============================================
# Top factors selected by RF importance (keep original interface)
#     (here we take the first 10 sorted by permutation importance)
# ============================================

up_top   = up_result_lstm["importance_series"].head(10).index.tolist()
down_top = down_result_lstm["importance_series"].head(10).index.tolist()

print("UP Top features (by permutation importance):",   up_top)
print("DOWN Top features (by permutation importance):", down_top)

UP Top features (by permutation importance): ['vol_32', 'vol_32_slope_25', 'mom_8_slope_10', 'vol_32_slope_10', 'vol_16_slope_25', 'mom_8_slope_5', 'vol_ma_10', 'ma_4_slope_5', 'mom_8_slope', 'ma_8_slope_10']
DOWN Top features (by permutation importance): ['ma_32_slope_25', 'vol_32_slope_10', 'close', 'vol_ma_20', 'hold_ret_10', 'hold_ma_bias_20', 'vol_32', 'ma_8_slope_25', 'ma_16_slope', 'high']


In [76]:
# ============================================
# 11. Feature preprocessing: fit(train) + transform(train/test)
# ============================================
from scipy.stats import spearmanr, linregress 

def infer_factor_sign(
    df,
    factor,
    future_col="future_ret_H",
    n_group=5,
    min_samples=300,
    bootstrap_n=200,
    min_confidence=0.7
):
    """
    Factor-direction inference:

    Components:
        1. Qcut bucketing (baseline)
        2. IC sign (linear direction)
        3. Monotonicity check
        4. Bootstrap direction stability
        5. Final direction decision

    Returns:
        final_sign: +1 / -1 / 0
        detail: dict with all intermediate diagnostics
    """

    # ======================
    # Prepare data
    # ======================
    sub = df[[factor, future_col]].dropna().copy()
    if len(sub) < min_samples:
        return 0, {"error": "too_few_samples"}

    x = sub[factor].values
    y = sub[future_col].values

    # ======================
    # 1) Qcut bucketing
    # ======================
    try:
        sub["group"] = pd.qcut(x, n_group, labels=False, duplicates="drop")
        g = sub.groupby("group")[future_col].mean()
        qcut_sign = np.sign(g.iloc[-1] - g.iloc[0])
    except Exception:
        qcut_sign = 0
        g = None  

    # ======================
    # 2) IC sign (information coefficient direction)
    # ======================
    IC = np.corrcoef(x, y)[0, 1]
    if np.isnan(IC):
        ic_sign = 0
    else:
        ic_sign = np.sign(IC)

    # ======================
    # 3) Monotonicity (Spearman)
    # ======================
    try:
        # Check monotonicity using group means from quantile buckets
        if g is None:
            mono_sign = 0
            spearman_r = np.nan
        else:
            group_returns = g.values
            group_index = np.arange(len(group_returns))
            spearman_r, _ = spearmanr(group_index, group_returns)
            mono_sign = np.sign(spearman_r) if not np.isnan(spearman_r) else 0
    except Exception:
        mono_sign = 0
        spearman_r = np.nan

    # ========================================
    # 3b) Linear regression slope direction + t-stat 
    # ========================================
    
    reg_sign = 0            
    reg_tstat = np.nan
    reg_slope = np.nan

    try:
        x_centered = x - np.mean(x)
        y_centered = y - np.mean(y)

        slope, intercept, r_value, p_value, std_err = linregress(x_centered, y_centered)
        reg_slope = slope

        if (std_err is None) or (std_err == 0) or np.isnan(std_err) or np.isnan(slope):
            reg_sign = 0
            reg_tstat = np.nan
        else:
            reg_sign = int(np.sign(slope))
            reg_tstat = slope / std_err
    except Exception:
        reg_sign = 0
        reg_tstat = np.nan
        reg_slope = np.nan

    # ======================
    # 4) Bootstrap stability
    # ======================
    signs = []
    n = len(sub)

    # Block bootstrap: each block ≈ 5% of the sample, at least 5 obs
    block_size = max(5, n // 20)
    n_blocks = int(np.ceil(n / block_size))

    for _ in range(bootstrap_n):
        idx_list = []
        for _ in range(n_blocks):
            start_max = n - block_size
            if start_max < 0:
                start_max = 0
            start = np.random.randint(0, start_max + 1)
            idx_block = range(start, min(start + block_size, n))
            idx_list.extend(idx_block)

        idx = np.array(idx_list[:n]) 
        xb = x[idx]
        yb = y[idx]

        # Bootstrap Qcut sign (same logic as main)
        try:
            dfb = pd.DataFrame({"f": xb, "r": yb}).dropna()
            dfb["g"] = pd.qcut(
                dfb["f"], n_group,
                labels=False, duplicates="drop"
            )
            gb = dfb.groupby("g")["r"].mean()
            bsign = np.sign(gb.iloc[-1] - gb.iloc[0])
        except Exception:
            bsign = 0

        signs.append(bsign)

    signs = np.array(signs)
    pos_prob = np.mean(signs == 1)
    neg_prob = np.mean(signs == -1)

    # Bootstrap direction 
    if pos_prob > neg_prob:
        bootstrap_sign = 1
    elif neg_prob > pos_prob:
        bootstrap_sign = -1
    else:
        bootstrap_sign = 0

    bootstrap_confidence = max(pos_prob, neg_prob)

    # ======================
    # 5) Final direction decision (weighted ensemble)
    # ======================

    # Weights 
    w_qcut = 0.35
    w_ic   = 0.35
    w_mono = 0.2
    w_boot = 0.1

    raw_score = (
        w_qcut * qcut_sign +
        w_ic   * ic_sign +
        w_mono * mono_sign +
        w_boot * bootstrap_sign
    )

    # Regression constraint: t-stat threshold + mild reinforcement weight
    reg_t_min = 2.0   # |t| ≥ 2 considered statistically significant
    w_reg = 0.15      # small boost, should not dominate the ensemble

    # Use regression slope direction as a hard constraint + consistency enhancer
    if not np.isnan(reg_tstat) and abs(reg_tstat) >= reg_t_min and reg_sign != 0:
        raw_sign_before_reg = np.sign(raw_score)

        if raw_sign_before_reg != 0:
            if raw_sign_before_reg != reg_sign:
                # Case 1: regression is significant but points to the opposite direction
                #         → veto the direction entirely
                raw_score = 0.0
            else:
                # Case 2: regression direction is consistent with ensemble direction
                #         → modestly reinforce raw_score
                raw_score = raw_score + w_reg * reg_sign
        # If raw_score == 0, keep it 0 and do not let regression alone decide direction

    final_sign = int(np.sign(raw_score))

    # Bootstrap confidence filter 
    if bootstrap_confidence < min_confidence:
        final_sign = 0  # if direction is unstable, prefer to drop the factor

    # ======================
    # Pack and return results
    # ======================
    detail = {
        "qcut_sign": int(qcut_sign),
        "ic": IC,
        "ic_sign": int(ic_sign),
        "spearman": spearman_r,
        "mono_sign": int(mono_sign),
        "bootstrap_sign": int(bootstrap_sign),
        "bootstrap_confidence": bootstrap_confidence,
        "final_sign_rawscore": raw_score,
        "final_sign": int(final_sign),

        # Regression diagnostics
        "reg_slope": float(reg_slope) if not np.isnan(reg_slope) else np.nan,
        "reg_tstat": float(reg_tstat) if not np.isnan(reg_tstat) else np.nan,
        "reg_sign": int(reg_sign),

        # Block bootstrap configuration
        "bootstrap_block_size": int(block_size),
        "bootstrap_n": int(bootstrap_n),
    }

    return final_sign, detail


In [77]:
# 1) Keep the original order from RF importance
up_factors = list(dict.fromkeys(up_top))
down_factors = list(dict.fromkeys(down_top))

print("Factors used for LONG (UP) sign inference:", up_factors)
print("Factors used for SHORT (DOWN) sign inference:", down_factors)
print("\n")

# 2) General helper: build sign_dict for a given factor list
def build_sign_dict(
    factor_list,
    train_df,
    future_col="future_ret_H",
    n_group=5,
    bootstrap_n=300,
    min_confidence=0.7
):
    """
    Build {factor -> sign} via infer_factor_sign on the training set.

    Returns
    -------
    sign_dict : dict
        {factor_name: +1 / -1 / 0}
    detail_dict : dict
        {factor_name: full diagnostics returned by infer_factor_sign}
    """
    sign_dict = {}
    detail_dict = {}
    for f in factor_list:
        sign, detail = infer_factor_sign(
            train_df,
            factor=f,
            future_col=future_col,
            n_group=n_group,
            bootstrap_n=bootstrap_n,
            min_confidence=min_confidence
        )
        sign = int(sign)
        sign_dict[f] = sign
        detail_dict[f] = detail
    return sign_dict, detail_dict


up_dict, up_detail = build_sign_dict(up_factors, train_all)
down_dict, down_detail = build_sign_dict(down_factors, train_all)


# 3) Filter by sign using the ORIGINAL factor order,
#    instead of arbitrary dict order
final_up_factors   = [f for f in up_factors   if up_dict.get(f, 0) != 0]
removed_up_factors = [f for f in up_factors   if up_dict.get(f, 0) == 0]

final_down_factors   = [f for f in down_factors if down_dict.get(f, 0) != 0]
removed_down_factors = [f for f in down_factors if down_dict.get(f, 0) == 0]

print("UP sign_dict:", up_dict)
print("DOWN sign_dict:", down_dict)
print("\n")
print("Final UP factors (non-zero sign):", final_up_factors)
print("Final DOWN factors (non-zero sign):", final_down_factors)


Factors used for LONG (UP) sign inference: ['vol_32', 'vol_32_slope_25', 'mom_8_slope_10', 'vol_32_slope_10', 'vol_16_slope_25', 'mom_8_slope_5', 'vol_ma_10', 'ma_4_slope_5', 'mom_8_slope', 'ma_8_slope_10']
Factors used for SHORT (DOWN) sign inference: ['ma_32_slope_25', 'vol_32_slope_10', 'close', 'vol_ma_20', 'hold_ret_10', 'hold_ma_bias_20', 'vol_32', 'ma_8_slope_25', 'ma_16_slope', 'high']


UP sign_dict: {'vol_32': -1, 'vol_32_slope_25': 1, 'mom_8_slope_10': -1, 'vol_32_slope_10': 1, 'vol_16_slope_25': 1, 'mom_8_slope_5': -1, 'vol_ma_10': -1, 'ma_4_slope_5': 1, 'mom_8_slope': 1, 'ma_8_slope_10': 1}
DOWN sign_dict: {'ma_32_slope_25': 0, 'vol_32_slope_10': 1, 'close': -1, 'vol_ma_20': -1, 'hold_ret_10': 0, 'hold_ma_bias_20': 0, 'vol_32': -1, 'ma_8_slope_25': 0, 'ma_16_slope': 1, 'high': -1}


Final UP factors (non-zero sign): ['vol_32', 'vol_32_slope_25', 'mom_8_slope_10', 'vol_32_slope_10', 'vol_16_slope_25', 'mom_8_slope_5', 'vol_ma_10', 'ma_4_slope_5', 'mom_8_slope', 'ma_8_slope_

In [78]:
# =============================
# 12. Score Function
# =============================
def add_score(
    df,
    up_factors,
    down_factors,
    up_sign_dict,
    down_sign_dict,
    lookback=50,
    winsor_limit=5,
    use_robust=True,
    remove_corr=True,
    corr_threshold=0.7,
    factor_group_map=None,      # ⚠ 当前版本中不再使用，所有因子等权
    conflict_check=True,
    conflict_threshold=0.1      # 单因子“强贡献”的阈值（按贡献值）
):
    """
    新版 add_score（无因子分组版本）：

    输入：
        up_factors      : 多头侧使用的因子名列表（UP 模型 TopN）
        down_factors    : 空头侧使用的因子名列表（DOWN 模型 TopN）
        up_sign_dict    : 因子在 UP 方向上的 sign（+1/-1/0）
        down_sign_dict  : 因子在 DOWN 方向上的 sign（+1/-1/0）

    输出（在 df 上新增三列）：
        score_long  : 多头得分（3 日平滑）
        score_short : 空头得分（3 日平滑）
        score       : score_long - score_short
    """

    df = df.copy()

    # =============================
    # 0. 清洗列表 & 按 sign 过滤
    # =============================
    # 去重、保序
    up_factors   = list(dict.fromkeys(up_factors))
    down_factors = list(dict.fromkeys(down_factors))

    # 处理“同一个因子在 up/down 里方向相反”的情况：直接剔除
    overlap = set(up_factors) & set(down_factors)
    conflict_drop = set()
    for f in overlap:
        su = np.sign(up_sign_dict.get(f, 0.0))
        sd = np.sign(down_sign_dict.get(f, 0.0))
        if su != 0 and sd != 0 and su != sd:
            print(f"[警告] 因子 {f} 在 up_sign_dict / down_sign_dict 中方向相反，将从多头/空头列表中移除。")
            conflict_drop.add(f)

    if conflict_drop:
        up_factors   = [f for f in up_factors   if f not in conflict_drop]
        down_factors = [f for f in down_factors if f not in conflict_drop]

    # 合并后的总因子列表（允许同一个因子同时出现在 up/down，只要方向一致）
    all_factors = list(dict.fromkeys(up_factors + down_factors))

    if len(all_factors) == 0:
        df["score"]       = np.nan
        df["score_long"]  = np.nan
        df["score_short"] = np.nan
        return df

    # =============================
    # 1. 自动剔除共线因子（在 all_factors 上做）
    # =============================
    if remove_corr and len(all_factors) > 1:
        existing = [f for f in all_factors if f in df.columns]
        if len(existing) > 1:
            corr = df[existing].corr().abs()
            keep = []
            for f in existing:
                if not keep:
                    keep.append(f)
                    continue
                if all(corr[f][k] < corr_threshold for k in keep):
                    keep.append(f)

            kept = set(keep)
            all_factors  = [f for f in all_factors  if f in kept]
            up_factors   = [f for f in up_factors   if f in kept]
            down_factors = [f for f in down_factors if f in kept]

    if len(all_factors) == 0:
        df["score"]       = np.nan
        df["score_long"]  = np.nan
        df["score_short"] = np.nan
        return df

    # =============================
    # 2. 计算 rolling z-score（robust 可选）
    # =============================
    valid_factors = []

    for f in all_factors:
        if f not in df.columns:
            print(f"[跳过因子] {f}: 不在 df 列中")
            continue

        series = df[f].astype(float)

        if use_robust:
            rolling_med = series.rolling(lookback).median()
            rolling_mad = (series - rolling_med).abs().rolling(lookback).median()
            z = (series - rolling_med) / (rolling_mad * 1.4826 + 1e-6)
        else:
            rolling_mean = series.rolling(lookback).mean()
            rolling_std  = series.rolling(lookback).std()
            z = (series - rolling_mean) / (rolling_std + 1e-6)

        # 如果全 NaN 或几乎无波动 → 跳过
        if z.isna().all() or z.std(skipna=True) == 0:
            print(f"[跳过因子] {f}: Z 值无效（可能全 NaN 或常数）")
            continue

        # winsorize
        z = z.clip(-winsor_limit, winsor_limit)

        df[f"_z_{f}"] = z.astype(float)
        valid_factors.append(f)

    if len(valid_factors) == 0:
        df["score"]       = np.nan
        df["score_long"]  = np.nan
        df["score_short"] = np.nan
        return df

    # 只保留 z 有效因子中的 up/down
    up_valid   = [f for f in up_factors   if f in valid_factors]
    down_valid = [f for f in down_factors if f in valid_factors]

    valid_union = list(dict.fromkeys(up_valid + down_valid))
    if len(valid_union) == 0:
        df["score"]       = np.nan
        df["score_long"]  = np.nan
        df["score_short"] = np.nan
        return df

    # =============================
    # 3. 构造 Z 矩阵 & 等权权重 & 方向
    # =============================
    # --- 3.1 Z 矩阵 ---
    z_cols = [f"_z_{f}" for f in valid_union]
    Z = df[z_cols].copy()
    Z.columns = valid_union

    # --- 3.2 所有有效因子等权（不再区分组）---
    n_factors = len(valid_union)
    if n_factors == 0:
        df["score"]       = np.nan
        df["score_long"]  = np.nan
        df["score_short"] = np.nan
        return df

    equal_weight = 1.0 / n_factors
    w_series = pd.Series({f: equal_weight for f in valid_union}, dtype=float)

    # --- 3.3 三套 sign：
    #   1) up_signs：只用于多头 score 计算
    #   2) down_signs：只用于空头 score 计算
    #   3) union_signs：用于冲突检测（合并后的方向）
    up_signs = pd.Series(
        {f: float(np.sign(up_sign_dict.get(f, 0.0))) for f in up_valid},
        dtype=float
    )
    down_signs = pd.Series(
        {f: float(np.sign(down_sign_dict.get(f, 0.0))) for f in down_valid},
        dtype=float
    )

    union_signs_dict = {}
    for f in valid_union:
        su = np.sign(up_sign_dict.get(f, 0.0))
        sd = np.sign(down_sign_dict.get(f, 0.0))

        if f in up_valid and f not in down_valid:
            s = su
        elif f in down_valid and f not in up_valid:
            s = sd
        else:
            # 同时出现在 up/down 且方向一致（方向相反的已在前面移除）
            s = su if su != 0 else sd
        union_signs_dict[f] = float(s)

    union_signs = pd.Series(union_signs_dict, dtype=float)

    # =============================
    # 4. 计算多头 / 空头原始得分
    # =============================

    # --- 4.1 多头 score：只看 up_valid + up_sign_dict ---
    if len(up_valid) > 0:
        Z_up = Z[up_valid]
        contrib_up = Z_up.mul(up_signs, axis=1)                # 正 = 看多，负 = 看空（以 UP 的方向定义）
        contrib_up_w = contrib_up.mul(w_series[up_valid], axis=1)
        bull_up = contrib_up_w.clip(lower=0.0)                 # 只取“牛市贡献”
        score_long_raw = bull_up.sum(axis=1)
    else:
        score_long_raw = pd.Series(0.0, index=df.index)

    # --- 4.2 空头 score：只看 down_valid + down_sign_dict ---
    if len(down_valid) > 0:
        Z_down = Z[down_valid]
        contrib_down = Z_down.mul(down_signs, axis=1)          # 正 = 看多，负 = 看空（以 DOWN 的方向定义）
        contrib_down_w = contrib_down.mul(w_series[down_valid], axis=1)
        bear_down = (-contrib_down_w).clip(lower=0.0)          # 只取“熊市贡献”（负贡献取绝对值）
        score_short_raw = bear_down.sum(axis=1)
    else:
        score_short_raw = pd.Series(0.0, index=df.index)

    # 净信号
    raw_score = score_long_raw - score_short_raw

    # =============================
    # 5. 冲突抑制（基于合并后的方向 union_signs）
    # =============================
    if conflict_check:
        # 用 union_signs + w_series 构造“总贡献矩阵”
        contrib_all = Z.mul(union_signs, axis=1)
        contrib_all_w = contrib_all.mul(w_series, axis=1)

        strong_pos = (contrib_all_w >  conflict_threshold).sum(axis=1)
        strong_neg = (contrib_all_w < -conflict_threshold).sum(axis=1)
        total_strong = strong_pos + strong_neg

        bull_mask = score_long_raw > score_short_raw
        bear_mask = score_short_raw > score_long_raw
        tie_mask  = score_long_raw == score_short_raw

        minority_limit = total_strong / 3.0

        bull_conflict = bull_mask & (strong_neg > minority_limit)
        bear_conflict = bear_mask & (strong_pos > minority_limit)
        tie_conflict  = tie_mask  & (total_strong > 0)

        conflict_mask = bull_conflict | bear_conflict | tie_conflict

        raw_score[conflict_mask]       = 0.0
        score_long_raw[conflict_mask]  = 0.0
        score_short_raw[conflict_mask] = 0.0

    # =============================
    # 6. 轻度平滑（CTA 常用）
    # =============================
    df["score_long"]  = score_long_raw.rolling(3).mean()
    df["score_short"] = score_short_raw.rolling(3).mean()
    df["score"]       = df["score_long"] - df["score_short"]

    return df







In [79]:
# ============================================
# 1) Call add_score to compute score_long / score_short / score
# ============================================
train_score = add_score(
    df=train_all,                 # factor DataFrame
    up_factors=final_up_factors,  # long-side factor list
    down_factors=final_down_factors,  # short-side factor list
    up_sign_dict=up_dict,         # long-side sign dictionary
    down_sign_dict=down_dict,     # short-side sign dictionary
    lookback=50,
    winsor_limit=3,
    use_robust=True,
    remove_corr=True,
    corr_threshold=0.7,
    conflict_check=True,
    conflict_threshold=0.1
)

# ============================================
# 2) Sanity checks: which factors are actually used / score distribution
# ============================================

# 2.1 Final effective factors 
z_cols = [c for c in train_score.columns if c.startswith("_z_")]
valid_factors = [c.replace("_z_", "") for c in z_cols]

print("\n[Check] Final list of effective factors used in scoring:")
print(valid_factors)
print(f"Total number of effective factors: {len(valid_factors)}")

# 2.2 Take a quick look at score_long / score_short / score
print("\n[Check] score_long / score_short / score (head):")
print(train_score[["score_long", "score_short", "score"]].head(10))

# 2.3 Summary statistics of score distribution (for choosing T later)
print("\n[Check] Distribution of score:")
print(train_score["score"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))



[Check] Final list of effective factors used in scoring:
['vol_32', 'vol_32_slope_25', 'mom_8_slope_10', 'vol_32_slope_10', 'mom_8_slope_5', 'vol_ma_10', 'ma_4_slope_5', 'mom_8_slope', 'ma_8_slope_10', 'close']
Total number of effective factors: 10

[Check] score_long / score_short / score (head):
            score_long  score_short  score
datetime                                  
2014-04-03         NaN          NaN    NaN
2014-04-04         NaN          NaN    NaN
2014-04-08         0.0          0.0    0.0
2014-04-09         0.0          0.0    0.0
2014-04-10         0.0          0.0    0.0
2014-04-11         0.0          0.0    0.0
2014-04-14         0.0          0.0    0.0
2014-04-15         0.0          0.0    0.0
2014-04-16         0.0          0.0    0.0
2014-04-17         0.0          0.0    0.0

[Check] Distribution of score:
count    1417.000000
mean        0.143941
std         0.213031
min        -0.336726
10%        -0.017905
25%         0.000000
50%         0.050857
75%  

In [80]:
# ============================================
# 1) Call add_score to compute score_long / score_short / score
# ============================================

test_score = add_score(
    df=test,                       # factor DataFrame for test sample
    up_factors=final_up_factors,   # long-side factor list
    down_factors=final_down_factors,  # short-side factor list
    up_sign_dict=up_dict,          # long-side sign dictionary
    down_sign_dict=down_dict,      # short-side sign dictionary
    lookback=50,
    winsor_limit=3,
    use_robust=True,
    remove_corr=True,
    corr_threshold=0.7,
    conflict_check=True,
    conflict_threshold=0.1
)

# ============================================
# 2) Sanity checks: which factors are actually used / score distribution
# ============================================

# 2.1 Final effective factors 
z_cols = [c for c in test_score.columns if c.startswith("_z_")]
valid_factors = [c.replace("_z_", "") for c in z_cols]

print("\n[Check] Final list of effective factors used in scoring:")
print(valid_factors)
print(f"Total number of effective factors: {len(valid_factors)}")

# 2.2 Quick look at score_long / score_short / score
print("\n[Check] score_long / score_short / score (head):")
print(test_score[["score_long", "score_short", "score"]].head(10))

# 2.3 Summary of score distribution (for choosing T later)
print("\n[Check] Distribution of score:")
print(test_score["score"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))



[Check] Final list of effective factors used in scoring:
['vol_32', 'vol_32_slope_25', 'mom_8_slope_10', 'vol_32_slope_10', 'mom_8_slope_5', 'vol_ma_10', 'ma_4_slope_5', 'mom_8_slope', 'ma_8_slope_10', 'close']
Total number of effective factors: 10

[Check] score_long / score_short / score (head):
            score_long  score_short  score
datetime                                  
2020-02-05         NaN          NaN    NaN
2020-02-06         NaN          NaN    NaN
2020-02-07         0.0          0.0    0.0
2020-02-10         0.0          0.0    0.0
2020-02-11         0.0          0.0    0.0
2020-02-12         0.0          0.0    0.0
2020-02-13         0.0          0.0    0.0
2020-02-14         0.0          0.0    0.0
2020-02-17         0.0          0.0    0.0
2020-02-18         0.0          0.0    0.0

[Check] Distribution of score:
count    945.000000
mean       0.103087
std        0.172983
min       -0.300705
10%       -0.014681
25%        0.000000
50%        0.000000
75%        0

In [82]:
print(train_score["score_long"].describe())
print(train_score["score_short"].describe())
print("mean long vs short:", train_score["score_long"].mean(), train_score["score_short"].mean())

print("pos rate:", (train_score["score"]>0).mean(), "neg rate:", (train_score["score"]<0).mean())


count    1417.000000
mean        0.192165
std         0.221999
min         0.000000
25%         0.000000
50%         0.123882
75%         0.314501
max         1.340697
Name: score_long, dtype: float64
count    1417.000000
mean        0.048224
std         0.080226
min         0.000000
25%         0.000000
50%         0.014571
75%         0.063773
max         0.621756
Name: score_short, dtype: float64
mean long vs short: 0.19216499044813867 0.04822445135729029
pos rate: 0.5186751233262861 neg rate: 0.14799154334038056


In [83]:
# ============================================
# 13. Choose a reasonable base_th 
# ============================================

def choose_base_th_from_train(
    train_df,
    close_col="close",
    base_th_ref=0.01,     # target cum_ret threshold when daily vol ≈ ref_vol (default 1%)
    vol_lookback=50,      # rolling window length for volatility
    ref_vol=0.01,         # reference “baseline volatility”, default 1%
    floor_mult=0.5,       # base_th ≥ base_th_ref * floor_mult
    ceil_mult=3.0,        # base_th ≤ base_th_ref * ceil_mult
    winsor_pct=1.0,       # winsorize both tails of daily returns by winsor_pct% (e.g. 1%)
    min_vol=0.002,        # lower bound for vol_median (e.g. 0.2%)
    max_vol=0.1,         # upper bound for vol_median (e.g. 5%)
    verbose=True
):
    """
    Logic:
        1. Use training close prices to compute daily returns `ret`.
           - Optionally winsorize ret at winsor_pct% in both tails (clip).
        2. Compute rolling volatility `rolling_vol` with window `vol_lookback`,
           then take its median `vol_median`.
           - Clamp vol_median into [min_vol, max_vol] if provided.
        3. Scale base_th_ref linearly according to vol_median vs ref_vol:
               base_th_data = base_th_ref * (vol_median / ref_vol)
        4. Apply floor & ceiling for extra robustness:
               base_th_floor = floor_mult * base_th_ref
               base_th_ceil  = ceil_mult  * base_th_ref
               base_th = min(max(base_th_data, base_th_floor), base_th_ceil)

    Parameters:
        train_df    : training DataFrame (must contain close_col)
        close_col   : name of the close price column
        base_th_ref : reference threshold (target cum_ret when daily vol ≈ ref_vol)
        vol_lookback: window length for rolling volatility
        ref_vol     : reference volatility (typically 0.01 for 1%)
        floor_mult  : base_th lower bound factor relative to base_th_ref
        ceil_mult   : base_th upper bound factor relative to base_th_ref
        winsor_pct  : percent for two-sided winsorization on ret
                      (None or 0 → no winsorization)
        min_vol     : lower bound for vol_median (None → no lower bound)
        max_vol     : upper bound for vol_median (None → no upper bound)
        verbose     : whether to print intermediate diagnostics

    Returns:
        base_th (float): can be directly passed to cta_backtest_money as base_th
    """
    # 1) Daily returns
    if close_col not in train_df.columns:
        raise KeyError(f"[choose_base_th_from_train] '{close_col}' not found in train_df columns.")

    close = train_df[close_col].astype(float)
    ret = close.pct_change().dropna()

    if len(ret) < vol_lookback:
        if verbose:
            print("[choose_base_th_from_train] Too few data points, returning base_th_ref directly.")
        return float(base_th_ref)

    # 1.5) Light winsorization on daily returns (heavy-tail robust)
    if winsor_pct is not None and winsor_pct > 0:
        # winsor_pct = 1 → 1% and 99% quantiles
        low_q, high_q = np.percentile(ret, [winsor_pct, 100 - winsor_pct])
        ret = ret.clip(low_q, high_q)

    # 2) Median of rolling volatility on training set
    rolling_vol = ret.rolling(vol_lookback).std()
    vol_median_raw = rolling_vol.median()

    if (vol_median_raw is None) or (not np.isfinite(vol_median_raw)) or (vol_median_raw <= 0):
        if verbose:
            print("[choose_base_th_from_train] vol_median invalid, returning base_th_ref directly.")
        return float(base_th_ref)

    vol_median = float(vol_median_raw)

    # 2.5) Clamp vol_median into a “reasonable” range
    #      If min_vol / max_vol are specified, clip vol_median into [min_vol, max_vol].
    vol_median_before_clip = vol_median

    if min_vol is not None:
        vol_median = max(vol_median, float(min_vol))
    if max_vol is not None:
        vol_median = min(vol_median, float(max_vol))

    # 3) Data-driven base_th scaled by volatility
    base_th_data = base_th_ref * (vol_median / ref_vol)

    # 4) floor / ceil protection
    base_th_floor = floor_mult * base_th_ref
    base_th_ceil  = ceil_mult  * base_th_ref

    base_th_clipped = max(base_th_data, base_th_floor)
    base_th_clipped = min(base_th_clipped, base_th_ceil)

    if verbose:
        print(
            "[choose_base_th_from_train] "
            f"vol_median_raw={vol_median_before_clip:.4f}, "
            f"vol_median_used={vol_median:.4f}, "
            f"base_th_data={base_th_data:.4f}, "
            f"floor={base_th_floor:.4f}, ceil={base_th_ceil:.4f} "
            f"→ base_th={base_th_clipped:.4f}"
        )

    return float(base_th_clipped)


In [84]:
# ============================================================
# Call choose_base_th_from_train on the training set
# ===========================================================
base_th = choose_base_th_from_train(
    train_df=train_all,
    close_col="close",     
    base_th_ref=0.01,      # target cum_ret threshold 1% when daily vol ≈ 1%
    vol_lookback=50,       # 50-day rolling volatility window
    ref_vol=0.01,          # reference volatility 1%
    floor_mult=0.6,        # lower bound ≈ 0.6% when base_th_ref = 1%
    ceil_mult=3.0,         # upper bound ≈ 3% when base_th_ref = 1%
    winsor_pct=1.0,        # winsorize both tails of daily returns at 1%
    min_vol=0.002,         # vol_median at least 0.2%
    max_vol=0.1,          # vol_median at most 5%
    verbose=True           # print detailed diagnostics
)

print(f"\n[Result] base_th selected from training set = {base_th:.6f}")


[choose_base_th_from_train] vol_median_raw=0.0124, vol_median_used=0.0124, base_th_data=0.0124, floor=0.0060, ceil=0.0300 → base_th=0.0124

[Result] base_th selected from training set = 0.012427


In [85]:
# ============================================
# 13. Choose a reasonable T
# ============================================

def choose_T_from_train(
    train_df,
    score_col="score",
    q=0.80,          # default: use the 80% quantile of |score| (much looser than 0.95)
    min_T=0.5,       # only trade when |score| is at least this strong
    max_T=None,      # optional: an upper cap to prevent T from becoming too large
    verbose=True
):
    """
    Choose a signal threshold T from the training sample, based on the score distribution.
    We will later use this T in rules like: open a position only if |score| > T.
    Design (aligned with your trading intuition):
        1. `score` itself is a directional signal:
               score > 0 → bullish bias
               score < 0 → bearish bias

        2. To avoid over-trading on weak signals, impose a strength threshold T:
               score >  T  → open long
               score < -T  → open short

        3. How T is chosen:
               - Data-driven part:
                     T_data = q-quantile of |score| (default q = 0.80,
                              which is much looser than q = 0.95)
               - Then enforce a fixed lower bound:
                     T = max(T_data, min_T)
               - If `max_T` is given, also enforce:
                     T = min(T, max_T)

           This way:
               - Always require “|score| > min_T (e.g. 0.7) before trading”
               - When the score distribution is very volatile, T can automatically
                 increase to avoid excessively frequent trades.
    """
    if score_col not in train_df.columns:
        raise KeyError(f"[choose_T_from_train] '{score_col}' not found in train_df columns.")

    s = train_df[score_col].dropna().astype(float)
    if len(s) == 0:
        raise ValueError("[choose_T_from_train] Training scores are all NaN; cannot estimate T.")

    # 1) Data-driven threshold: q-quantile of |score|
    T_data = s.abs().quantile(q)

    # 2) Enforce at least min_T (e.g. 0.7)
    T_final = max(T_data, float(min_T))

    # 3) Optional upper cap
    if max_T is not None:
        T_final = min(T_final, float(max_T))

    if verbose:
        msg = (
            f"[choose_T_from_train] "
            f"T_data(q={q:.2f})={T_data:.4f}, "
            f"min_T={min_T:.4f}"
        )
        if max_T is not None:
            msg += f", max_T={max_T:.4f}"
        msg += f" → T={T_final:.4f}"
        print(msg)

    return float(T_final)


In [86]:
T = choose_T_from_train(train_df=train_score,score_col="score", q=0.8, min_T=0.5)

s = train_score["score"].dropna().astype(float)
print("P(|score|>T):", (s.abs() > T).mean())
print("P(score> T):", (s > T).mean())
print("P(score<-T):", (s < -T).mean())
print("share of zeros:", (s.abs() < 1e-12).mean())


[choose_T_from_train] T_data(q=0.80)=0.3172, min_T=0.5000 → T=0.5000
P(|score|>T): 0.0748059280169372
P(score> T): 0.0748059280169372
P(score<-T): 0.0
share of zeros: 0.3323923782639379


In [87]:
# ================================
# Select T_train on the training set 
# ================================

T_train = choose_T_from_train(
    train_df=train_score,
    score_col="score",  
    q=0.8,              # use the 85% quantile of |score| as reference
    min_T=0.2,           # only trade when |score| is at least this strong 
    max_T=None,          # no upper cap; set e.g. 2.0 if you want one
    verbose=True        
)

print(f"\n[Result] T_train selected from training set = {T_train:.4f}")


[choose_T_from_train] T_data(q=0.80)=0.3172, min_T=0.2000 → T=0.3172

[Result] T_train selected from training set = 0.3172


In [91]:
import pandas as pd

def align_weekly_from_test_start(
    test_df: pd.DataFrame,
    weekly_df: pd.DataFrame
):
    """
    根据日线 test 数据中的“第一天日期”，在周线数据中找到对应的那一周，
    并从该周开始截取 weekly_df。

    要求：
    - test_df.index   为 DatetimeIndex（日线日期）
    - weekly_df.index 为 DatetimeIndex（周线日期，通常是每周五的 W-FRI）

    参数
    ----
    test_df : pd.DataFrame
        日线 test 数据（例如从 load_main_csv 分割出来的 test）。
    weekly_df : pd.DataFrame
        周线数据（例如从 FG_weekly_clean.csv 读出后再 set_index 的 DataFrame）。

    返回
    ----
    weekly_trim : pd.DataFrame
        从 start_week_date 开始（含）截取后的周线数据，
        后续你在这个基础上构建周线筛选因子即可。
    first_test_date : pd.Timestamp
        test_df 中最早那一行的日期（index.min）。
    start_week_date : pd.Timestamp
        weekly_df 中第一个 >= first_test_date 的周线日期，
        即“包含 first_test_date 的那一周”的周终日（比如那周的周五）。
    """

    # 1) 检查 index 类型
    if not isinstance(test_df.index, pd.DatetimeIndex):
        raise ValueError("test_df 的 index 必须是 DatetimeIndex（请确认你已经 set_index('datetime')）。")

    if not isinstance(weekly_df.index, pd.DatetimeIndex):
        raise ValueError("weekly_df 的 index 必须是 DatetimeIndex（请对周线数据做类似 set_index('datetime') 的处理）。")

    # 2) test 的第一天日期
    first_test_date = test_df.index.min()

    # 3) 在周线里找“包含这一天的那一周”
    #    对于 W-FRI 周线：某个 daily_date d 所属的周线日期
    #    就是第一个 >= d 的那条周线 index。
    weekly_after = weekly_df[weekly_df.index >= first_test_date]

    if weekly_after.empty:
        raise ValueError(
            f"weekly_df 中不存在 >= first_test_date ({first_test_date}) 的周线数据。"
        )

    start_week_date = weekly_after.index[0]

    # 4) 从该周开始截取周线
    weekly_trim = weekly_after.copy()

    return weekly_trim, first_test_date, start_week_date

In [95]:
weekly_trim, first_test_date, start_week_date = align_weekly_from_test_start(
    test_df=test,
    weekly_df=data2
)

print("test 第一日:", first_test_date)
print("周线起始周:", start_week_date)
print("截取后的周线形状:", weekly_trim.shape)


test 第一日: 2020-02-05 00:00:00
周线起始周: 2020-02-07 00:00:00
截取后的周线形状: (200, 7)


In [96]:
def build_weekly_filters(
    weekly_df: pd.DataFrame,
    price_col: str = "close",
    hold_col: str = "hold",
    vol_window: int = 4,          # 周波动的窗口，默认 4 周
    trend_fast: int = 13,         # 快均线（趋势）
    trend_slow: int = 26,         # 慢均线（趋势）
    crowd_lookback: int = 4,      # 拥挤度回看周数
    crowd_hold_thresh: float = 0.2,   # 持仓 4 周涨幅阈值（20%）
    crowd_price_thresh: float = 0.05, # 价格 4 周涨幅阈值（5%）
    use_prev_week: bool = True    # 默认用“上一周”的因子做过滤，防止泄漏
) -> pd.DataFrame:
    """
    在周线数据上构建 CTA 用的“周线过滤因子”。

    输入参数
    --------
    weekly_df : pd.DataFrame
        周线数据，index 建议为 DatetimeIndex，至少包含：
            - price_col (默认 "close")
            - hold_col  (默认 "hold")
        一般来自你之前构造好的 *_weekly_clean.csv （再 set_index）。
    price_col : str
        周线收盘价列名，用来计算收益 / 均线。
    hold_col : str
        周线持仓量列名。
    vol_window : int
        周波动率 rolling 的窗口长度（单位：周）。
    trend_fast, trend_slow : int
        周线趋势快 / 慢均线的窗口长度。
    crowd_lookback : int
        判断持仓拥挤度的回看周数。
    crowd_hold_thresh : float
        持仓拥挤：hold 在 crowd_lookback 周内涨幅超过该比例视为“持仓大幅增加”。
    crowd_price_thresh : float
        价格在 crowd_lookback 周内涨幅超过该比例视为“价格明显上涨”。
    use_prev_week : bool
        - True : 输出的因子全部 shift(1)，表示“本周使用上周的周度信息（无未来）”；
        - False: 因子基于当周数据（偏激进，有一定未来信息成分）。

    返回
    ----
    weekly_factors : pd.DataFrame
        index 与 weekly_df 对齐，包含以下列（前缀统一用 wk_）：
            - wk_ret           : 周收益率
            - wk_ma_fast       : 快均线
            - wk_ma_slow       : 慢均线
            - wk_trend_raw     : MA_fast - MA_slow（原始多空差）
            - wk_trend_sign    : 趋势方向 (+1/-1/0)
            - wk_vol           : 周波动率 (rolling std)
            - wk_vol_regime    : 波动 regime (0=低,1=中,2=高)
            - wk_hold_chg      : 持仓周变化 (同比上周)
            - wk_hold_chg_L    : crowd_lookback 周持仓变化
            - wk_price_chg_L   : crowd_lookback 周价格涨跌
            - wk_crowded_long  : 多头拥挤标记 (bool)
    """

    if not isinstance(weekly_df.index, pd.DatetimeIndex):
        raise ValueError("build_weekly_filters: weekly_df 的 index 必须是 DatetimeIndex。")

    if price_col not in weekly_df.columns:
        raise ValueError(f"build_weekly_filters: weekly_df 中缺少价格列 '{price_col}'。")

    if hold_col not in weekly_df.columns:
        raise ValueError(f"build_weekly_filters: weekly_df 中缺少持仓列 '{hold_col}'。")

    w = weekly_df.sort_index().copy()

    # ======================
    # 1) 周收益 & 周波动
    # ======================
    w_ret = w[price_col].pct_change()
    w_vol = w_ret.rolling(vol_window, min_periods=vol_window//2).std()

    # 按整体分位数划分波动 regime
    # 0: 低波  1: 中波  2: 高波
    q_low = w_vol.quantile(0.3)
    q_high = w_vol.quantile(0.7)

    def classify_vol(v):
        if np.isnan(v):
            return np.nan
        if v <= q_low:
            return 0
        elif v >= q_high:
            return 2
        else:
            return 1

    w_vol_regime = w_vol.apply(classify_vol)

    # ======================
    # 2) 周线趋势 (MA_fast / MA_slow)
    # ======================
    ma_fast = w[price_col].rolling(trend_fast, min_periods=trend_fast//2).mean()
    ma_slow = w[price_col].rolling(trend_slow, min_periods=trend_slow//2).mean()
    trend_raw = ma_fast - ma_slow

    trend_sign = np.sign(trend_raw)
    # 对非常小的差值，可选：视为 0（去噪）
    eps = 1e-8
    trend_sign = trend_sign.where(trend_raw.abs() > eps, 0.0)

    # ======================
    # 3) 持仓拥挤度（多头）
    # ======================
    hold = w[hold_col].astype(float)

    # 周度持仓变化
    hold_chg = hold.diff()

    # crowd_lookback 周持仓变化
    hold_chg_L = hold - hold.shift(crowd_lookback)

    # crowd_lookback 周价格涨跌
    price = w[price_col].astype(float)
    price_chg_L = price / price.shift(crowd_lookback) - 1.0

    # 持仓基期避免除 0
    hold_base = hold.shift(crowd_lookback)
    with np.errstate(divide="ignore", invalid="ignore"):
        hold_pct_chg_L = np.where(
            hold_base > 0,
            hold_chg_L / hold_base,
            np.nan
        )

    # 多头拥挤：持仓大幅增加 + 价格持续上涨
    crowded_long = (
        (hold_pct_chg_L > crowd_hold_thresh) &
        (price_chg_L > crowd_price_thresh)
    )

    # ======================
    # 4) 组合成一个 DataFrame
    # ======================
    weekly_factors = pd.DataFrame(index=w.index)

    weekly_factors["wk_ret"]          = w_ret
    weekly_factors["wk_vol"]          = w_vol
    weekly_factors["wk_vol_regime"]   = w_vol_regime

    weekly_factors["wk_ma_fast"]      = ma_fast
    weekly_factors["wk_ma_slow"]      = ma_slow
    weekly_factors["wk_trend_raw"]    = trend_raw
    weekly_factors["wk_trend_sign"]   = trend_sign

    weekly_factors["wk_hold_chg"]     = hold_chg
    weekly_factors["wk_hold_chg_L"]   = hold_chg_L
    weekly_factors["wk_price_chg_L"]  = price_chg_L
    weekly_factors["wk_crowded_long"] = crowded_long.astype("float")  # bool -> float(0/1)，方便后面运算

    # ======================
    # 5) 是否使用“上一周”的周度信息
    # ======================
    if use_prev_week:
        weekly_factors = weekly_factors.shift(1)

    return weekly_factors

In [97]:
weekly_filters = build_weekly_filters(
    weekly_df=weekly_trim,
    price_col="close",
    hold_col="hold",
    use_prev_week=True
)

print(weekly_filters.head(10))

              wk_ret    wk_vol  wk_vol_regime   wk_ma_fast  wk_ma_slow  \
datetime                                                                 
2020-02-07       NaN       NaN            NaN          NaN         NaN   
2020-02-14       NaN       NaN            NaN          NaN         NaN   
2020-02-21  0.009013       NaN            NaN          NaN         NaN   
2020-02-28  0.017418  0.005944            0.0          NaN         NaN   
2020-03-06 -0.066725  0.046345            2.0          NaN         NaN   
2020-03-13  0.007996  0.039328            1.0          NaN         NaN   
2020-03-20 -0.101260  0.057711            2.0  3978.684377         NaN   
2020-03-27 -0.081516  0.047727            2.0  3876.840396         NaN   
2020-04-03 -0.069531  0.047871            2.0  3772.073399         NaN   
2020-04-10  0.051640  0.069121            2.0  3708.023483         NaN   

            wk_trend_raw  wk_trend_sign  wk_hold_chg  wk_hold_chg_L  \
datetime                                

In [98]:
def attach_weekly_filters_to_daily(daily_df, weekly_filters):
    """
    weekly_filters: index = 周线日期 (W-FRI)，列 = wk_*
    daily_df     : index = 日线日期
    逻辑：对日线做一个 reindex + ffill，让每一天携带最近一条周线因子。
    """
    # 按日频率重采样周度因子
    wf = weekly_filters.copy()
    wf_daily = wf.reindex(daily_df.index, method="bfill")
    # 合并
    merged = daily_df.join(wf_daily)
    return merged


In [99]:
test_with_weekly = attach_weekly_filters_to_daily(test_score, weekly_filters)

# 之后把 test_with_weekly 传入你的 cta_backtest_money，
# 并在回测函数内部根据 wk_trend_sign / wk_vol_regime / wk_crowded_long
# 去调节是否开仓、仓位大小、base_th 等。

In [100]:
# ============================================
# 15. Backtest
# ============================================

def backtest_money(
    df,
    score_col="score",
    T=None,                 # Must be provided externally (e.g., output of choose_T_from_train)
    base_th=0.01,           # choose_base_th_from_train 
    max_hold_days=10,       # Maximum holding period (10 days)
    init_capital=500000,    # Initial capital 500k
    pos_pct=0.3,            # Use 30% of current equity as margin for each trade
    leverage=10,            # Directional leverage
    shift_signal=True,      # Whether to execute the signal on the next bar

    # ====== Weekly filter parameters ======
    use_weekly_filter=True,
    wk_trend_col="wk_trend_sign",
    wk_vol_regime_col="wk_vol_regime",
    wk_crowded_long_col="wk_crowded_long",
    block_vol_regime=2,
    allow_trend_zero_open=False,

    # ====== Performance statistics parameters (NEW) ======
    rf_annual=0.012,        # Annual risk-free rate, default 1.2%
    annual_trading_days=252 # Number of trading days used for annualized Sharpe
):
    

    df = df.copy()

    # === 0) Underlying daily return ===
    df["ret"] = df["close"].pct_change().fillna(0.0)

    if T is None:
        raise ValueError(
            "[cta_backtest_money] Parameter T is not specified. "
            "Please first compute T on the training set using choose_T_from_train (or similar), "
            "and then pass it into this function."
        )

    # === 1) Raw directional signal ===
    raw_score = df[score_col]

    sig = pd.Series(0.0, index=df.index, dtype=float)
    sig[raw_score >  T] =  1.0
    sig[raw_score < -T] = -1.0

    if shift_signal:
        # Signal takes effect on the next bar to avoid trading on the same bar
        # using a score computed from that bar’s close.
        sig = sig.shift(1).fillna(0.0)

    df["signal_raw"] = sig

    # === 1.5) Weekly filters: only affect whether opening a position is allowed ===
    weekly_ok = pd.Series(True, index=df.index, dtype=bool)

    if use_weekly_filter:
        # ---- Trend filter: wk_trend_sign ----
        if wk_trend_col in df.columns:
            trend = df[wk_trend_col]

            if not allow_trend_zero_open:
                weekly_ok &= (trend != 0).fillna(False)

            same_dir_or_zero = ((trend * sig) >= 0) | (sig == 0)
            weekly_ok &= same_dir_or_zero.fillna(False)

        # ---- Volatility filter: wk_vol_regime ----
        if wk_vol_regime_col in df.columns:
            vol_reg = df[wk_vol_regime_col]
            high_vol_mask = (vol_reg == block_vol_regime)
            weekly_ok &= ~(high_vol_mask & (sig != 0))

        # ---- Crowding filter: wk_crowded_long ----
        if wk_crowded_long_col in df.columns:
            crowded = df[wk_crowded_long_col].fillna(0.0)
            block_long = (crowded >= 0.5) & (sig > 0)
            weekly_ok &= ~block_long

    sig_eff = sig.copy()
    sig_eff[~weekly_ok] = 0.0
    df["signal_eff"] = sig_eff
    df["weekly_ok"] = weekly_ok.astype(float)

    # === 2) Initialize strategy state ===
    capital = init_capital
    pos_dir = 0.0
    hold_value = 0.0
    cum_ret = 0.0
    hold_days = 0
    reached_tp1 = False

    capital_list = []
    pos_exposure_list = []
    hold_value_list = []

    trade_returns = []   # >>> NEW: record per-trade margin return cum_ret

    # === 3) Main loop: update day by day ===
    for i in range(len(df)):
        daily_ret_raw = df["ret"].iloc[i]
        signal_today = df["signal_eff"].iloc[i]

        # --- A. When flat → check if we should open a new position ---
        if pos_dir == 0.0 and signal_today != 0.0:
            pos_dir = signal_today
            hold_value = capital * pos_pct
            cum_ret = 0.0
            hold_days = 0
            reached_tp1 = False

        # --- B. Daily P&L (real capital) ---
        effective_notional = pos_dir * leverage * hold_value
        daily_profit = effective_notional * daily_ret_raw
        capital += daily_profit

        # --- C. While in position, update cumulative return for the trade ---
        if pos_dir != 0.0:
            cum_ret += pos_dir * leverage * daily_ret_raw
            hold_days += 1

            if cum_ret >= base_th:
                reached_tp1 = True

            # --- D. Exit conditions ---
            exit_flag = False
            if cum_ret >= 2 * base_th:                      # Hard take-profit
                exit_flag = True
            elif cum_ret <= -0.9 * base_th:                   # Stop loss
                exit_flag = True
            elif reached_tp1 and cum_ret <= 0.65 * base_th:  # Pullback take-profit
                exit_flag = True
            elif hold_days >= max_hold_days:                # Time stop
                exit_flag = True

            if exit_flag:
                # >>> NEW: record this trade's margin return
                trade_returns.append(cum_ret)

                pos_dir = 0.0
                hold_value = 0.0
                cum_ret = 0.0
                hold_days = 0
                reached_tp1 = False

        # --- E. Record daily state ---
        capital_list.append(capital)
        pos_exposure_list.append(pos_dir * leverage)
        hold_value_list.append(hold_value)

    # === 4) Write back result columns (equity curve) ===
    df["capital"] = capital_list
    df["pos"] = pos_exposure_list
    df["hold_value"] = hold_value_list
    df["equity"] = df["capital"] / float(init_capital)

    # === 5) Compute statistics and attach to attrs (NEW) ===
    # 5.1 Portfolio daily returns & annualized Sharpe
    equity_series = pd.Series(capital_list, index=df.index)
    port_ret = equity_series.pct_change().fillna(0.0)

    # Convert annual risk-free rate to daily
    rf_daily = (1.0 + rf_annual) ** (1.0 / annual_trading_days) - 1.0
    excess_ret = port_ret - rf_daily

    if excess_ret.std() > 0:
        sharpe_annual = (
            excess_ret.mean() / excess_ret.std()
        ) * np.sqrt(annual_trading_days)
    else:
        sharpe_annual = np.nan

    # 5.2 Trade-level statistics
    trade_returns = np.asarray(trade_returns, dtype=float)
    n_trades = int(trade_returns.size)
    n_win = int((trade_returns > 0).sum())
    n_loss = int((trade_returns < 0).sum())
    win_rate = float(n_win / n_trades) if n_trades > 0 else np.nan

    if n_loss > 0:
        # Larger losses are more negative; take the smallest five
        worst_losses = np.sort(trade_returns[trade_returns < 0])
        worst5_loss_ret = worst_losses[:5].tolist()
    else:
        worst5_loss_ret = []

    bt_stats = {
        "n_trades": n_trades,
        "n_win": n_win,
        "n_loss": n_loss,
        "win_rate": win_rate,
        "worst5_loss_ret": worst5_loss_ret,
        "sharpe_annual": sharpe_annual,
        "rf_annual": rf_annual,
        "annual_trading_days": annual_trading_days,
    }

    df.attrs["bt_stats"] = bt_stats 

    return df

In [101]:
bt_rb = backtest_money(
    df=test_with_weekly,
    score_col="score",
    T= 0.226,
    base_th=0.012427,
    max_hold_days=10,
    init_capital=500000,
    pos_pct=0.5,
    leverage=10,
    shift_signal=True,
    use_weekly_filter=True      
)

In [46]:
df_bt= backtest_money(df=test_with_weekly, T=0.226, base_th=0.012427, max_loss_margin_pct=1.10, maint_margin_ratio=0.70,verbose=True, print_on_entry=False)
trade_log = df_bt.attrs["trade_log"]   # 每笔交易完整表
bt_stats  = df_bt.attrs["bt_stats"]


NameError: name 'test_with_weekly' is not defined

In [47]:
#########################################
# Plot cumulative returns 
#########################################
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import numpy as np

# ===== 0) Backtest stats from attrs =====
stats = bt_rb.attrs.get("bt_stats", {})

n_trades   = stats.get("n_trades", np.nan)
n_win      = stats.get("n_win", np.nan)
n_loss     = stats.get("n_loss", np.nan)
win_rate   = stats.get("win_rate", np.nan)
worst5_ret = stats.get("worst5_loss_ret", [])
sharpe     = stats.get("sharpe_annual", np.nan)
rf_annual  = stats.get("rf_annual", 0.012)

# formatted strings
win_rate_str = "N/A" if np.isnan(win_rate) else f"{win_rate * 100:.2f}%"
sharpe_str   = "N/A" if np.isnan(sharpe)   else f"{sharpe:.3f}"
rf_str       = f"{rf_annual * 100:.2f}%"

if worst5_ret:
    worst5_str = ", ".join([f"{x * 100:.1f}%" for x in worst5_ret])
else:
    worst5_str = "N/A"

# ===== 1) Cumulative return (RB) =====
ret_rb = bt_rb["equity"] / bt_rb["equity"].iloc[0] - 1
ret_rb = ret_rb.dropna()

# ===== 2) Figure & axis =====
fig, ax = plt.subplots(figsize=(12, 5))


fig.subplots_adjust(right=0.76)

ax.plot(ret_rb.index, ret_rb, label="PTA", linewidth=1.4)

ax.set_title("PTA Strategy Cumulative Return (Pure Terephthalic Acid)", fontsize=14)
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Cumulative Return", fontsize=12)

# y-axis in percentage
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

# x-axis: yearly ticks
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=30)

# horizontal zero line
ax.axhline(0.0, color="grey", linestyle="--", linewidth=0.8, alpha=0.7)

# light grid
ax.grid(True, linestyle="--", alpha=0.35)


for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# legend in upper left
ax.legend(
    loc="upper left",
    frameon=False,
    title="Contract",
)

# ===== 3) Stats text on the right =====
x_stats = 0.79  
y_top   = 0.86
line_h  = 0.055  

fig.text(x_stats, y_top, "Backtest Summary", ha="left", va="top",
         fontsize=11, fontweight="bold")

fig.text(x_stats, y_top - 1*line_h,
         f"Total trades: {n_trades}", ha="left", va="top", fontsize=9)
fig.text(x_stats, y_top - 2*line_h,
         f"Winning trades: {n_win}", ha="left", va="top", fontsize=9)
fig.text(x_stats, y_top - 3*line_h,
         f"Losing trades: {n_loss}", ha="left", va="top", fontsize=9)
fig.text(x_stats, y_top - 4*line_h,
         f"Hit ratio: {win_rate_str}", ha="left", va="top", fontsize=9)

fig.text(x_stats, y_top - 5.2*line_h,
         "Worst 5 trades (margin R):", ha="left", va="top", fontsize=9)
fig.text(x_stats, y_top - 6.2*line_h,
         worst5_str, ha="left", va="top", fontsize=9)

fig.text(x_stats, y_top - 7.4*line_h,
         f"Annual Sharpe (rf={rf_str}): {sharpe_str}",
         ha="left", va="top", fontsize=9)

plt.show()


NameError: name 'bt_rb' is not defined

In [48]:
# ===== 4) Save figure as PNG for download =====
output_path = "pta_xgb.png"   
fig.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to: {output_path}")


NameError: name 'fig' is not defined